In [1]:

import os

files = os.listdir('/content')

for file in files:
    if file.endswith('.csv'):
        print(file)


matches.csv
ball_by_ball.csv
teams.csv
competitions.csv
playing_eleven.csv
competition_team_player_mapping.csv


In [2]:

import pandas as pd

# Load all datasets
ball = pd.read_csv('/content/ball_by_ball.csv')
matches = pd.read_csv('/content/matches.csv')
competitions = pd.read_csv('/content/competitions.csv')
players = pd.read_csv('/content/competition_team_player_mapping.csv')
playing_xi = pd.read_csv('/content/playing_eleven.csv')
teams = pd.read_csv('/content/teams.csv')

# Check dataset size and columns
datasets = {
    "Ball By Ball": ball,
    "Matches": matches,
    "Competitions": competitions,
    "Players Mapping": players,
    "Playing XI": playing_xi,
    "Teams": teams
}

for name, df in datasets.items():
    print("\n==========", name, "==========")
    print("Rows and Columns:", df.shape)
    print("Column Names:", list(df.columns))
    print("Missing Values:", df.isnull().sum().sum())

# Show sample ball-by-ball data
display(ball.head(5))



========== Ball By Ball ==========
Rows and Columns: (200999, 33)
Column Names: ['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']
Missing Values: 766466

========== Matches ==========
Rows and Columns: (883, 22)
Column Names: ['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 

,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC
2,13511,1,1,3,3,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Square Cut,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
3,13511,1,1,4,4,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Backfoot defensive,Inswing,Over,On-side,2022-01-23 00:00:00 UTC
4,13511,1,1,5,5,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Pull shot,Seamed In,Over,On-side,2022-01-23 00:00:00 UTC


In [3]:

# Identify IPL and T20I competitions

ipl = competitions[
    competitions['comp_name'].str.contains('IPL', case=False, na=False)
]

t20i = competitions[
    competitions['match_format'] == 'T20I'
]

print("IPL Competitions:", len(ipl))
display(ipl[['comp_id', 'comp_name', 'season']])

print("T20I Competitions:", len(t20i))

print("\nIPL Seasons:")
print(sorted(ipl['season'].unique()))

print("\nT20I Seasons:")
print(sorted(t20i['season'].unique()))


IPL Competitions: 5


,comp_id,comp_name,season
12,1358,IPL 2022,2022
13,1715,IPL 2023,2023
15,2193,IPL 2024,2024
16,2725,IPL 2025,2025
18,3298,IPL 2026,2026


T20I Competitions: 134

IPL Seasons:
['2022', '2023', '2024', '2025', '2026']

T20I Seasons:
['2022', '2022/23', '2023', '2023/24', '2024', '2024/25', '2025', '2026']


In [4]:

# Step 5: Filter IPL and T20I matches

# Standardize IDs
competitions['comp_id'] = competitions['comp_id'].astype(str)
matches['comp_id'] = matches['comp_id'].astype(str)
matches['match_id'] = matches['match_id'].astype(str)
ball['match_id'] = ball['match_id'].astype(str)

# Identify IPL and T20I competitions
target_comps = competitions[
    competitions['comp_name'].str.contains('IPL', case=False, na=False) |
    (competitions['match_format'] == 'T20I')
]

# Select only relevant matches
selected_matches = matches[
    matches['comp_id'].isin(target_comps['comp_id'])
].copy()

# Attach competition information
selected_matches = selected_matches.merge(
    target_comps,
    on='comp_id',
    how='left'
)

# Select ball-by-ball records for these matches
selected_ball = ball[
    ball['match_id'].isin(selected_matches['match_id'])
].copy()

# Results
print("Total Selected Competitions:", len(target_comps))
print("Total Selected Matches:", len(selected_matches))
print("Total Selected Deliveries:", len(selected_ball))

print("\nMatch Format Distribution:")
print(selected_matches['match_format'].value_counts())

print("\nSeason Distribution:")
print(selected_matches['season'].value_counts().sort_index())

print("\nSample Selected Matches:")
display(selected_matches.head())


Total Selected Competitions: 139
Total Selected Matches: 853
Total Selected Deliveries: 194014

Match Format Distribution:
match_format
T20I    482
T20     371
Name: count, dtype: int64

Season Distribution:
season
2022       168
2022/23     12
2023       137
2023/24      3
2024       183
2024/25      3
2025       201
2026       146
Name: count, dtype: int64

Sample Selected Matches:


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result,comp_name,match_format,season
0,13511,1331,1,2022-01-22,League,54,West Indies,28,West Indies,22,...,0,NaN,NaN,NaN,0,28.0,West Indies Won by 9 Wickets,England in West Indies T20I Series 2022,T20I,2022
1,13512,1331,2,2022-01-23,League,54,West Indies,28,West Indies,22,...,0,NaN,NaN,NaN,0,22.0,England Won by 1 Run,England in West Indies T20I Series 2022,T20I,2022
2,13513,1331,3,2022-01-26,League,54,West Indies,28,West Indies,22,...,4986,NaN,NaN,NaN,0,28.0,West Indies Won by 20 runs,England in West Indies T20I Series 2022,T20I,2022
3,13514,1331,4,2022-01-29,League,54,West Indies,28,West Indies,22,...,0,NaN,NaN,NaN,0,22.0,England Won by 34 runs,England in West Indies T20I Series 2022,T20I,2022
4,13515,1331,5,2022-01-30,League,54,West Indies,28,West Indies,22,...,0,NaN,NaN,NaN,0,28.0,West Indies Won by 17 runs,England in West Indies T20I Series 2022,T20I,2022


In [5]:

# Step 6: Explore ball-by-ball records

print("Total Deliveries:", len(selected_ball))

print("\n1. Normal Deliveries:")
display(selected_ball[
    ['striker_name', 'bowler_name', 'innings',
     'overid', 'runs', 'is_four', 'is_six']
].head(10))

print("\n2. Wicket Deliveries:")
display(selected_ball[
    selected_ball['is_wicket'] == True
][['striker_name', 'bowler_name', 'runs',
   'is_wicket', 'dismissal_type']].head(10))

print("\n3. Extras Deliveries:")
display(selected_ball[
    selected_ball['extras_type'].notna()
][['striker_name', 'bowler_name', 'runs',
   'extras_type', 'extras1_runs', 'extras_2_runs']].head(10))

print("\n4. Total Wickets:",
      selected_ball['is_wicket'].sum())

print("\n5. Total Fours:",
      selected_ball['is_four'].sum())

print("\n6. Total Sixes:",
      selected_ball['is_six'].sum())


Total Deliveries: 194014

1. Normal Deliveries:


,striker_name,bowler_name,innings,overid,runs,is_four,is_six
0,Jason Roy,Sheldon Cottrell,1,1,0,False,False
1,Jason Roy,Sheldon Cottrell,1,1,0,False,False
2,Jason Roy,Sheldon Cottrell,1,1,0,False,False
3,Jason Roy,Sheldon Cottrell,1,1,0,False,False
4,Jason Roy,Sheldon Cottrell,1,1,6,False,True
5,Jason Roy,Sheldon Cottrell,1,1,0,False,False
6,Tom Banton,Jason Holder,1,2,0,False,False
7,Tom Banton,Jason Holder,1,2,0,False,False
8,Tom Banton,Jason Holder,1,2,0,False,False
9,Tom Banton,Jason Holder,1,2,4,True,False



2. Wicket Deliveries:


,striker_name,bowler_name,runs,is_wicket,dismissal_type
5,Jason Roy,Sheldon Cottrell,0,True,Bowled
10,Tom Banton,Jason Holder,0,True,Caught
11,Moeen Ali,Jason Holder,0,True,Caught
29,James Vince,Sheldon Cottrell,0,True,Caught
42,Sam Billings,Akeal Hosein,0,True,Stump
63,Liam Dawson,Akeal Hosein,0,True,RunOut
72,Eoin Morgan,Romario Shepherd,0,True,Caught
101,Chris Jordan,Fabian Allen,0,True,Caught
119,Saqib Mahmood,Jason Holder,0,True,Caught
120,Adil Rashid,Jason Holder,0,True,Bowled



3. Extras Deliveries:


,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs
39,Eoin Morgan,Akeal Hosein,0,wide,2,0
43,Eoin Morgan,Romario Shepherd,0,wide,2,0
44,Eoin Morgan,Romario Shepherd,0,wide,2,0
143,Brandon King,Adil Rashid,0,legbyes,3,0
158,Shai Hope,Liam Dawson,0,legbyes,3,0
278,Moeen Ali,Kieron Pollard,0,wide,2,0
281,Moeen Ali,Kieron Pollard,0,wide,2,0
292,Moeen Ali,Jason Holder,0,wide,2,0
307,Sam Billings,Kieron Pollard,0,wide,2,0
320,Eoin Morgan,Kieron Pollard,0,wide,2,0



4. Total Wickets: 10486

5. Total Fours: 22669

6. Total Sixes: 12387


In [6]:

# Step 7: Check important match and ball columns

print("BALL-BY-BALL COLUMNS:")
print(selected_ball.columns.tolist())

print("\nMATCH DATA COLUMNS:")
print(selected_matches.columns.tolist())

print("\nSAMPLE MATCH SITUATION:")
display(selected_matches[[
    'match_id',
    'match_date',
    'comp_name',
    'home_team',
    'away_team',
    'result',
    'revised_target'
]].head(10))

print("\nSAMPLE BALL DATA:")
display(selected_ball.head(10))


BALL-BY-BALL COLUMNS:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

MATCH DATA COLUMNS:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result', 'comp_name', 'match_format', 'season']

SAMPLE MATCH SITUATION:


,match_id,match_date,comp_name,home_team,away_team,result,revised_target
0,13511,2022-01-22,England in West Indies T20I Series 2022,West Indies,England,West Indies Won by 9 Wickets,NaN
1,13512,2022-01-23,England in West Indies T20I Series 2022,West Indies,England,England Won by 1 Run,NaN
2,13513,2022-01-26,England in West Indies T20I Series 2022,West Indies,England,West Indies Won by 20 runs,NaN
3,13514,2022-01-29,England in West Indies T20I Series 2022,West Indies,England,England Won by 34 runs,NaN
4,13515,2022-01-30,England in West Indies T20I Series 2022,West Indies,England,West Indies Won by 17 runs,NaN
5,13522,2022-02-16,West Indies in India T20I Series 2022,India,West Indies,India Won by 6 Wickets,NaN
6,13523,2022-02-18,West Indies in India T20I Series 2022,India,West Indies,India Won by 8 Runs,NaN
7,13524,2022-02-20,West Indies in India T20I Series 2022,India,West Indies,India Won by 17 runs,NaN
8,13612,2022-02-11,Sri Lanka in Australia T20I Series 2022,Australia,Sri Lanka,Australia Won by 20 runs (D/L method),143.0
9,13613,2022-02-13,Sri Lanka in Australia T20I Series 2022,Australia,Sri Lanka,Australia Won the one over Eliminator,NaN



SAMPLE BALL DATA:


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC
2,13511,1,1,3,3,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Square Cut,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
3,13511,1,1,4,4,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Backfoot defensive,Inswing,Over,On-side,2022-01-23 00:00:00 UTC
4,13511,1,1,5,5,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Pull shot,Seamed In,Over,On-side,2022-01-23 00:00:00 UTC
5,13511,1,1,6,6,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0.0,Pull shot,Full Toss,Over,Off-side,2022-01-23 00:00:00 UTC
6,13511,1,2,1,7,22,28,6273,Tom Banton,1210,...,NaN,6,0,Pace,0.0,Backfoot defensive,Out Swing,Over,Off-side,2022-01-23 00:00:00 UTC
7,13511,1,2,2,8,22,28,6273,Tom Banton,1210,...,NaN,6,0,Pace,0.0,NaN,Out Swing,Over,Off-side,2022-01-23 00:00:00 UTC
8,13511,1,2,3,9,22,28,6273,Tom Banton,1210,...,NaN,6,0,Pace,0.0,Frontfoot defensive,Out Swing,Over,Off-side,2022-01-23 00:00:00 UTC
9,13511,1,2,4,10,22,28,6273,Tom Banton,1210,...,NaN,6,0,Pace,0.0,On drive,NaN,Over,Off-side,2022-01-23 00:00:00 UTC


In [7]:

# Step 7A: Inspect extras and dismissal types

print("EXTRAS TYPES:")
print(selected_ball['extras_type'].value_counts(dropna=False))

print("\nDISMISSAL TYPES:")
print(selected_ball['dismissal_type'].value_counts(dropna=False))

print("\nINNINGS DISTRIBUTION:")
print(selected_ball['innings'].value_counts())

print("\nSAMPLE EXTRAS RECORDS:")
display(
    selected_ball[
        selected_ball['extras_type'].notna()
    ][[
        'match_id', 'innings', 'overid',
        'runs', 'extras_type',
        'extras1_runs', 'extras_2_runs'
    ]].head(15)
)


EXTRAS TYPES:
extras_type
NaN         182548
wide          7474
legbyes       2661
Overstep       565
byes           505
other          260
Penalty          1
Name: count, dtype: int64

DISMISSAL TYPES:
dismissal_type
NaN                      183528
Caught                     7332
Bowled                     1690
LBW                         666
RunOut                      573
Stump                       200
Hit Wicket                   17
Retire Out                    6
Obstracting the field         1
Obstructing the field         1
Name: count, dtype: int64

INNINGS DISTRIBUTION:
innings
1    102054
2     91934
3        19
4         7
Name: count, dtype: int64

SAMPLE EXTRAS RECORDS:


,match_id,innings,overid,runs,extras_type,extras1_runs,extras_2_runs
39,13511,1,7,0,wide,2,0
43,13511,1,8,0,wide,2,0
44,13511,1,8,0,wide,2,0
143,13511,2,4,0,legbyes,3,0
158,13511,2,7,0,legbyes,3,0
278,13512,1,10,0,wide,2,0
281,13512,1,10,0,wide,2,0
292,13512,1,12,0,wide,2,0
307,13512,1,14,0,wide,2,0
320,13512,1,16,0,wide,2,0


In [8]:

# Step 7B: Inspect special extras and innings

print("OVERSTEP RECORDS:")
display(
    selected_ball[
        selected_ball['extras_type'] == 'Overstep'
    ][[
        'match_id', 'innings', 'overid',
        'runs', 'extras1_runs', 'extras_2_runs',
        'is_wicket', 'dismissal_type'
    ]].head(10)
)

print("\nOTHER EXTRAS RECORDS:")
display(
    selected_ball[
        selected_ball['extras_type'] == 'other'
    ][[
        'match_id', 'innings', 'overid',
        'runs', 'extras1_runs', 'extras_2_runs',
        'is_wicket', 'dismissal_type'
    ]].head(10)
)

print("\nINNINGS 3 AND 4 MATCHES:")
display(
    selected_ball[
        selected_ball['innings'].isin([3, 4])
    ][['match_id', 'innings', 'overid']].drop_duplicates()
)


OVERSTEP RECORDS:


,match_id,innings,overid,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
495,13513,1,4,0,1,0,False,NaN
1229,13515,2,20,1,1,0,False,NaN
1485,13523,1,1,0,1,4,False,NaN
1486,13523,1,1,1,1,0,False,NaN
3191,14094,1,1,0,1,0,False,NaN
4169,14098,1,1,0,1,0,False,NaN
4181,14098,1,3,0,1,0,False,NaN
4187,14098,1,4,4,1,0,False,NaN
4192,14098,1,5,1,1,0,False,NaN
4452,14099,1,6,0,1,0,False,NaN



OTHER EXTRAS RECORDS:


,match_id,innings,overid,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,0,5,0,False,NaN
1079,13515,1,16,0,5,0,False,NaN
1226,13515,2,19,2,5,0,False,NaN
2588,13614,1,18,0,5,4,False,NaN
2609,13614,2,2,0,5,0,False,NaN
3309,14094,1,19,6,5,0,False,NaN
4413,14098,2,20,0,5,0,False,NaN
5012,14101,1,17,0,5,0,False,NaN
5365,14102,2,19,0,5,0,True,RunOut
6575,14107,2,16,2,5,0,False,NaN



INNINGS 3 AND 4 MATCHES:


,match_id,innings,overid
105762,34429,3,1
105766,34429,4,1
131758,40725,3,1
131764,40725,4,1
155312,45512,3,1
155318,45512,4,1
185247,50359,3,1
185250,50359,4,1


In [10]:

# Check all column names
print("Columns in df:")
print(df.columns.tolist())

# Check dataset shape
print("\nDataset Shape:", df.shape)

# Check first 5 rows
display(df.head())


Columns in df:
['comp_id', 'team_id', 'team_name']

Dataset Shape: (353, 3)


,comp_id,team_id,team_name
0,1331,22,England
1,1331,28,West Indies
2,1335,18,India
3,1335,28,West Indies
4,1345,20,Australia


In [11]:

# Check all DataFrames currently available
import pandas as pd

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}: {obj.shape}")
        print(obj.columns.tolist())



ball: (200999, 33)
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

matches: (883, 22)
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']

competitions: (141, 4)
['comp_id', 'comp_name', 'match_format', 'season']

playe

In [12]:

# Step 6: Dataset Validation

ball = selected_ball.copy()

print("TOTAL DELIVERIES:", len(ball))
print("TOTAL MATCHES:", ball['match_id'].nunique())

print("\nINNINGS DISTRIBUTION:")
print(ball['innings'].value_counts().sort_index())

print("\nEXTRAS DISTRIBUTION:")
print(ball['extras_type'].value_counts(dropna=False))

print("\nDISMISSAL TYPES:")
print(ball['dismissal_type'].value_counts(dropna=False))

print("\nTOTAL WICKETS:", ball['is_wicket'].sum())

print("\nDUPLICATE ROWS:", ball.duplicated().sum())

print("\nMISSING VALUES:")
print(ball.isnull().sum())


TOTAL DELIVERIES: 194014
TOTAL MATCHES: 837

INNINGS DISTRIBUTION:
innings
1    102054
2     91934
3        19
4         7
Name: count, dtype: int64

EXTRAS DISTRIBUTION:
extras_type
NaN         182548
wide          7474
legbyes       2661
Overstep       565
byes           505
other          260
Penalty          1
Name: count, dtype: int64

DISMISSAL TYPES:
dismissal_type
NaN                      183528
Caught                     7332
Bowled                     1690
LBW                         666
RunOut                      573
Stump                       200
Hit Wicket                   17
Retire Out                    6
Obstracting the field         1
Obstructing the field         1
Name: count, dtype: int64

TOTAL WICKETS: 10486

DUPLICATE ROWS: 0

MISSING VALUES:
match_id                    0
innings                     0
overid                      0
over_ball_num               0
innings_ball_num            0
batting_team_id             0
bowling_team_id             0
striker_id 

In [13]:

# Find matches missing from ball dataset

match_ids = set(selected_matches['match_id'].unique())
ball_ids = set(selected_ball['match_id'].unique())

missing_ids = match_ids - ball_ids

print("Matches in selected_matches:", len(match_ids))
print("Matches in selected_ball:", len(ball_ids))
print("Missing matches:", len(missing_ids))

# Show missing match details
missing_matches = selected_matches[
    selected_matches['match_id'].isin(missing_ids)
]

display(missing_matches[
    ['match_id', 'comp_name', 'match_date',
     'home_team', 'away_team', 'match_format']
])


Matches in selected_matches: 853
Matches in selected_ball: 837
Missing matches: 16


,match_id,comp_name,match_date,home_team,away_team,match_format
396,31281,IPL 2024,2024-05-13,Gujarat Titans,Kolkata Knight Riders,T20
399,31284,IPL 2024,2024-05-16,Sunrisers Hyderabad,Gujarat Titans,T20
403,31288,IPL 2024,2024-05-19,Rajasthan Royals,Kolkata Knight Riders,T20
425,32610,Pakistan in England T20I Series 2024,2024-05-22,England,Pakistan,T20I
427,32612,Pakistan in England T20I Series 2024,2024-05-28,England,Pakistan,T20I
447,32622,Netherlands Tri-Nation T20I Series 2024,2024-05-20,Ireland,Scotland,T20I
453,38797,Pakistan in South Africa T20I Series 2024,2024-12-14,South Africa,Pakistan,T20I
481,36564,Australia in England T20I Series 2024,2024-09-15,England,Australia,T20I
498,36210,South Africa in England T20I Series 2025,2025-09-14,England,South Africa,T20I
578,40761,IPL 2025,2025-05-17,Royal Challengers Bengaluru,Kolkata Knight Riders,T20


In [14]:

# Check innings 3 and 4 records

special_innings = selected_ball[
    selected_ball['innings'].isin([3, 4])
]

print("Special innings deliveries:", len(special_innings))

display(
    special_innings[
        ['match_id', 'innings', 'overid',
         'striker_name', 'bowler_name', 'runs',
         'is_wicket', 'dismissal_type']
    ].sort_values(['match_id', 'innings', 'overid'])
)


Special innings deliveries: 26


,match_id,innings,overid,striker_name,bowler_name,runs,is_wicket,dismissal_type
105762,34429,3,1,Kusal Mendis,Washington Sundar,0,False,NaN
105763,34429,3,1,Kusal Mendis,Washington Sundar,1,False,NaN
105764,34429,3,1,Kusal Perera,Washington Sundar,0,True,Caught
105765,34429,3,1,Pathum Nissanka,Washington Sundar,0,True,Caught
105766,34429,4,1,Suryakumar Yadav,Maheesh Theekshana,4,False,NaN
131758,40725,3,1,Shimron Hetmyer,Mitchell Starc,0,False,NaN
131759,40725,3,1,Shimron Hetmyer,Mitchell Starc,4,False,NaN
131760,40725,3,1,Shimron Hetmyer,Mitchell Starc,1,False,NaN
131761,40725,3,1,Riyan Parag,Mitchell Starc,4,False,NaN
131762,40725,3,1,Riyan Parag,Mitchell Starc,0,True,RunOut


In [15]:

# Check missing matches in ORIGINAL ball dataset

original_missing = ball[
    ball['match_id'].isin(missing_ids)
]

print("Records found in original ball:", len(original_missing))

print("\nMatch-wise delivery count:")

print(
    original_missing.groupby('match_id')
    .size()
    .reindex(sorted(missing_ids), fill_value=0)
)

print("\nAvailable missing match IDs:")
print(sorted(original_missing['match_id'].unique()))


Records found in original ball: 0

Match-wise delivery count:
match_id
31281    0
31284    0
31288    0
32610    0
32612    0
32622    0
36210    0
36564    0
38797    0
40761    0
41502    0
41503    0
42030    0
46008    0
48513    0
50189    0
dtype: int64

Available missing match IDs:
[]


In [16]:

# Separate normal innings and Super Overs

normal_ball = selected_ball[
    selected_ball['innings'].isin([1, 2])
].copy()

super_over_ball = selected_ball[
    selected_ball['innings'].isin([3, 4])
].copy()

print("Normal innings deliveries:", len(normal_ball))
print("Super Over deliveries:", len(super_over_ball))

print("\nSuper Over matches:")
print(sorted(super_over_ball['match_id'].unique()))

print("\nNormal innings matches:", normal_ball['match_id'].nunique())


Normal innings deliveries: 193988
Super Over deliveries: 26

Super Over matches:
['34429', '40725', '45512', '50359']

Normal innings matches: 837


In [17]:

original_missing = ball[
    ball['match_id'].isin(missing_ids)
]

print("Records in original ball:", len(original_missing))
print("Matches found:", original_missing['match_id'].nunique())

print(
    original_missing.groupby('match_id').size()
)



Records in original ball: 0
Matches found: 0
Series([], dtype: int64)


In [18]:

analysis_ball = selected_ball[
    selected_ball['innings'].isin([1, 2])
].copy()

super_over_analysis = selected_ball[
    selected_ball['innings'].isin([3, 4])
].copy()

analysis_matches = selected_matches[
    selected_matches['match_id'].isin(
        analysis_ball['match_id'].unique()
    )
].copy()

print("FINAL ANALYSIS DATA")
print("Deliveries:", len(analysis_ball))
print("Matches:", analysis_ball['match_id'].nunique())
print("Match metadata:", len(analysis_matches))
print("Super Over deliveries:", len(super_over_analysis))
print("Missing matches:", len(missing_ids))


FINAL ANALYSIS DATA
Deliveries: 193988
Matches: 837
Match metadata: 837
Super Over deliveries: 26
Missing matches: 16


In [19]:

# STEP 7.1: Innings Summary

innings_summary = (
    analysis_ball
    .groupby(['match_id', 'innings'])
    .agg(
        batting_team_id=('batting_team_id', 'first'),
        total_runs=('runs', 'sum'),
        total_wickets=('is_wicket', 'sum'),
        total_deliveries=('match_id', 'size'),
        total_overs=('overid', 'nunique')
    )
    .reset_index()
)

print("TOTAL INNINGS:", len(innings_summary))

print("\nINNINGS SUMMARY:")
display(innings_summary.head(15))

print("\nTOTAL MATCHES:", innings_summary['match_id'].nunique())

print("\nAVERAGE FIRST INNINGS SCORE:")
print(
    innings_summary[innings_summary['innings'] == 1]
    ['total_runs'].mean()
)

print("\nAVERAGE SECOND INNINGS SCORE:")
print(
    innings_summary[innings_summary['innings'] == 2]
    ['total_runs'].mean()
)


TOTAL INNINGS: 1652

INNINGS SUMMARY:


,match_id,innings,batting_team_id,total_runs,total_wickets,total_deliveries,total_overs
0,13511,1,22,100,10,121,20
1,13511,2,28,99,1,103,18
2,13512,1,22,163,8,128,20
3,13512,2,28,158,8,124,20
4,13513,1,28,211,5,125,20
5,13513,2,22,192,9,127,20
6,13514,1,22,182,6,128,20
7,13514,2,28,150,5,125,20
8,13515,1,28,168,4,127,20
9,13515,2,22,150,10,127,20



TOTAL MATCHES: 837

AVERAGE FIRST INNINGS SCORE:
165.57467144563918

AVERAGE SECOND INNINGS SCORE:
150.30184049079756


In [20]:

# Check innings count per match

match_innings_count = (
    innings_summary.groupby('match_id')['innings']
    .nunique()
)

print("Innings count distribution:")
print(match_innings_count.value_counts().sort_index())

# Matches with fewer than 2 normal innings
incomplete_matches = match_innings_count[
    match_innings_count < 2
]

print("\nMatches with fewer than 2 innings:",
      len(incomplete_matches))

# Show incomplete match details
incomplete_details = analysis_matches[
    analysis_matches['match_id'].isin(
        incomplete_matches.index
    )
]

display(
    incomplete_details[
        ['match_id', 'comp_name', 'match_date',
         'home_team', 'away_team', 'result']
    ]
)

# Check any matches with more than 2 innings
print("\nMatches with more than 2 innings:")
print(match_innings_count[match_innings_count > 2])


Innings count distribution:
innings
1     22
2    815
Name: count, dtype: int64

Matches with fewer than 2 innings: 22


,match_id,comp_name,match_date,home_team,away_team,result
94,15410,South Africa in India T20I Series 2022,2022-06-19,India,South Africa,Match Abandoned
95,15546,Bangladesh in West Indies T20I Series 2022,2022-07-02,West Indies,Bangladesh,No Result
234,20226,IPL 2023,2023-05-03,Lucknow Super Giants,Chennai Super Kings,Match Abandoned
281,21406,New Zealand in Pakistan T20I Series 2023,2023-04-20,Pakistan,New Zealand,Match Abandoned
315,28333,Bangladesh in New Zealand T20I Series 2023,2023-12-29,New Zealand,Bangladesh,No Result
420,31707,New Zealand in Pakistan T20I Series 2024,2024-04-18,Pakistan,New Zealand,No Result
464,37353,England in West Indies T20I Series 2024,2024-11-18,West Indies,England,No Result
488,35424,Netherlands Tri-Nation T20I Series [Aug 2024] ...,2024-08-24,Canada,United States Of America,No Result
565,40748,IPL 2025,2025-05-05,Sunrisers Hyderabad,Delhi Capitals,No Result
568,40751,IPL 2025,2025-05-08,Punjab Kings,Delhi Capitals,No Result



Matches with more than 2 innings:
Series([], Name: innings, dtype: int64)


In [21]:

# Identify matches having both innings

complete_match_ids = (
    innings_summary.groupby('match_id')['innings']
    .nunique()
)

complete_match_ids = complete_match_ids[
    complete_match_ids == 2
].index

# Create complete match datasets
complete_matches = analysis_matches[
    analysis_matches['match_id'].isin(complete_match_ids)
].copy()

complete_ball = analysis_ball[
    analysis_ball['match_id'].isin(complete_match_ids)
].copy()

# Separate first and second innings
first_innings = complete_ball[
    complete_ball['innings'] == 1
].copy()

second_innings = complete_ball[
    complete_ball['innings'] == 2
].copy()

print("Complete Matches:", len(complete_matches))
print("Complete Ball Records:", len(complete_ball))
print("First Innings Deliveries:", len(first_innings))
print("Second Innings Deliveries:", len(second_innings))

print("\nMatch Results:")
print(complete_matches['result'].value_counts(dropna=False))


Complete Matches: 815
Complete Ball Records: 192465
First Innings Deliveries: 100531
Second Innings Deliveries: 91934

Match Results:
result
India Won by 7 wickets                    8
Delhi Capitals Won by 6 wickets           7
Kolkata Knight Riders Won by 8 wickets    6
Gujarat Titans Won by 7 wickets           5
India Won by 6 Wickets                    5
                                         ..
Lucknow Super Giants won by 5 runs        1
Delhi Capitals Won by 15 runs             1
Chennai Super Kings Won by 77 runs        1
Lucknow Super Giants won by 1 run         1
Rajasthan Royals Won by 9 Wickets         1
Name: count, Length: 642, dtype: int64


In [22]:

# Check how runs and extras are stored

print("EXTRAS BREAKDOWN:")

extras_check = (
    complete_ball[
        complete_ball['extras_type'].notna()
    ]
    .groupby('extras_type')
    .agg(
        deliveries=('runs', 'size'),
        runs_sum=('runs', 'sum'),
        extras1_sum=('extras1_runs', 'sum'),
        extras2_sum=('extras_2_runs', 'sum')
    )
    .reset_index()
)

display(extras_check)

print("\nSAMPLE EXTRA DELIVERIES:")

display(
    complete_ball[
        complete_ball['extras_type'].notna()
    ][
        ['match_id', 'innings', 'overid',
         'striker_name', 'runs', 'extras_type',
         'extras1_runs', 'extras_2_runs']
    ].head(15)
)


EXTRAS BREAKDOWN:


,extras_type,deliveries,runs_sum,extras1_sum,extras2_sum
0,Overstep,558,824,558,48
1,Penalty,1,0,7,0
2,byes,505,0,2020,0
3,legbyes,2639,0,7917,0
4,other,256,403,1280,34
5,wide,7427,0,14854,0



SAMPLE EXTRA DELIVERIES:


,match_id,innings,overid,striker_name,runs,extras_type,extras1_runs,extras_2_runs
39,13511,1,7,Eoin Morgan,0,wide,2,0
43,13511,1,8,Eoin Morgan,0,wide,2,0
44,13511,1,8,Eoin Morgan,0,wide,2,0
143,13511,2,4,Brandon King,0,legbyes,3,0
158,13511,2,7,Shai Hope,0,legbyes,3,0
278,13512,1,10,Moeen Ali,0,wide,2,0
281,13512,1,10,Moeen Ali,0,wide,2,0
292,13512,1,12,Moeen Ali,0,wide,2,0
307,13512,1,14,Sam Billings,0,wide,2,0
320,13512,1,16,Eoin Morgan,0,wide,2,0


In [23]:

# Check unique values for each extras type

for extra in ['wide', 'Overstep', 'byes',
              'legbyes', 'other', 'Penalty']:

    print(f"\n--- {extra} ---")

    display(
        complete_ball[
            complete_ball['extras_type'] == extra
        ][
            ['runs', 'extras1_runs', 'extras_2_runs']
        ]
        .value_counts()
        .reset_index(name='count')
        .head(15)
    )



--- wide ---


,runs,extras1_runs,extras_2_runs,count
0,0,2,0,7427



--- Overstep ---


,runs,extras1_runs,extras_2_runs,count
0,0,1,0,212
1,1,1,0,166
2,4,1,0,79
3,6,1,0,45
4,2,1,0,33
5,0,1,1,12
6,0,1,4,9
7,3,1,0,2



--- byes ---


,runs,extras1_runs,extras_2_runs,count
0,0,4,0,505


... (2 outputs left out)


--- other ---


,runs,extras1_runs,extras_2_runs,count
0,0,5,0,110
1,1,5,0,48
2,4,5,0,40
3,6,5,0,26
4,2,5,0,18
5,0,5,4,7
6,0,5,1,6
7,3,5,0,1



--- Penalty ---


,runs,extras1_runs,extras_2_runs,count
0,0,7,0,1


In [24]:

# Check all distinct combinations of extras fields
print("Unique extras combinations:")

display(
    complete_ball[
        complete_ball['extras_type'].notna()
    ][
        ['extras_type', 'extras1_runs', 'extras_2_runs']
    ]
    .drop_duplicates()
    .sort_values(['extras_type', 'extras1_runs', 'extras_2_runs'])
)

# Check rows where second extras field is non-zero
print("\nRows with extras_2_runs > 0:")

display(
    complete_ball[
        complete_ball['extras_2_runs'] > 0
    ][
        ['match_id', 'innings', 'overid', 'runs',
         'extras_type', 'extras1_runs', 'extras_2_runs']
    ].head(20)
)

# Check normal deliveries
print("\nNormal deliveries:")

display(
    complete_ball[
        complete_ball['extras_type'].isna()
    ][
        ['runs', 'extras1_runs', 'extras_2_runs']
    ].value_counts().reset_index(name='count').head(15)
)


Unique extras combinations:


,extras_type,extras1_runs,extras_2_runs
495,Overstep,1,0
24453,Overstep,1,1
1485,Overstep,1,4
34203,Penalty,7,0
414,byes,4,0
143,legbyes,3,0
532,other,5,0
18844,other,5,1
2588,other,5,4
39,wide,2,0



Rows with extras_2_runs > 0:


,match_id,innings,overid,runs,extras_type,extras1_runs,extras_2_runs
1485,13523,1,1,0,Overstep,1,4
2588,13614,1,18,0,other,5,4
7607,14112,1,11,0,Overstep,1,4
11386,14127,2,9,0,other,5,4
18844,14159,2,9,0,other,5,1
24453,15796,2,14,0,Overstep,1,1
28010,16424,2,6,0,Overstep,1,4
36854,17555,1,5,0,NaN,6,1
47440,20186,2,20,0,Overstep,1,1
47803,20188,1,19,0,other,5,1



Normal deliveries:


,runs,extras1_runs,extras_2_runs,count
0,1,6,0,68855
1,0,6,0,64934
2,4,6,0,22402
3,6,6,0,12240
4,2,6,0,12002
5,3,6,0,623
6,5,6,0,22
7,0,6,1,1


In [25]:

print("BALL DATA COLUMNS:")
print(complete_ball.columns.tolist())

print("\nMATCH DATA COLUMNS:")
print(complete_matches.columns.tolist())

print("\nEXTRAS FIELD DATA TYPES:")
print(complete_ball[
    ['runs', 'extras1_runs', 'extras_2_runs', 'extras_type']
].dtypes)

print("\nONE MATCH SAMPLE:")
display(
    complete_ball[
        complete_ball['match_id'] == 13511
    ][
        ['innings', 'overid', 'runs', 'extras_type',
         'extras1_runs', 'extras_2_runs']
    ].head(25)
)


BALL DATA COLUMNS:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

MATCH DATA COLUMNS:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result', 'comp_name', 'match_format', 'season']

EXTRAS FIELD DATA TYPES:
runs            

,innings,overid,runs,extras_type,extras1_runs,extras_2_runs


In [26]:

print("First 10 available match IDs:")
print(complete_ball['match_id'].drop_duplicates().head(10).tolist())

# Select first available match
sample_match_id = complete_ball['match_id'].iloc[0]

print("\nSample Match ID:", sample_match_id)

display(
    complete_ball[
        complete_ball['match_id'] == sample_match_id
    ][
        ['innings', 'overid', 'over_ball_num',
         'runs', 'extras_type', 'extras1_runs',
         'extras_2_runs', 'is_wicket']
    ].head(30)
)

print("\nMatch metadata:")
display(
    complete_matches[
        complete_matches['match_id'] == sample_match_id
    ][
        ['match_id', 'comp_name', 'result',
         'revised_target', 'first_inn_revised_overs',
         'second_inn_revised_overs']
    ]
)


First 10 available match IDs:
['13511', '13512', '13513', '13514', '13515', '13522', '13523', '13524', '13612', '13613']

Sample Match ID: 13511


,innings,overid,over_ball_num,runs,extras_type,extras1_runs,extras_2_runs,is_wicket
0,1,1,1,0,NaN,6,0,False
1,1,1,2,0,NaN,6,0,False
2,1,1,3,0,NaN,6,0,False
3,1,1,4,0,NaN,6,0,False
4,1,1,5,6,NaN,6,0,False
5,1,1,6,0,NaN,6,0,True
6,1,2,1,0,NaN,6,0,False
7,1,2,2,0,NaN,6,0,False
8,1,2,3,0,NaN,6,0,False
9,1,2,4,4,NaN,6,0,False



Match metadata:


,match_id,comp_name,result,revised_target,first_inn_revised_overs,second_inn_revised_overs
0,13511,England in West Indies T20I Series 2022,West Indies Won by 9 Wickets,NaN,NaN,NaN


In [27]:

sample = complete_ball[
    complete_ball['match_id'] == '13511'
].copy()

print("Innings-wise batter runs:")
print(sample.groupby('innings')['runs'].sum())

print("\nInnings-wise delivery counts:")
print(sample.groupby('innings').size())

print("\nInnings-wise extras category counts:")
display(
    sample.groupby(['innings', 'extras_type'], dropna=False)
    .size()
    .reset_index(name='deliveries')
)

print("\nTeam details:")
display(
    sample.groupby('innings').agg(
        batting_team=('batting_team_id', 'first'),
        batter_runs=('runs', 'sum'),
        wickets=('is_wicket', 'sum')
    ).reset_index()
)


Innings-wise batter runs:
innings
1    100
2     99
Name: runs, dtype: int64

Innings-wise delivery counts:
innings
1    121
2    103
dtype: int64

Innings-wise extras category counts:


,innings,extras_type,deliveries
0,1,wide,3
1,1,NaN,118
2,2,legbyes,2
3,2,NaN,101



Team details:


,innings,batting_team,batter_runs,wickets
0,1,22,100,10
1,2,28,99,1


In [28]:

# Check whether extras_2_runs stores actual extra runs

print("All non-zero extras_2_runs records:")

display(
    complete_ball[
        complete_ball['extras_2_runs'] > 0
    ][
        ['match_id', 'innings', 'runs',
         'extras_type', 'extras1_runs',
         'extras_2_runs']
    ].head(30)
)

print("\nNon-zero extras_2_runs distribution:")
print(
    complete_ball['extras_2_runs']
    .value_counts()
    .sort_index()
)

print("\nExtras type and extras_2_runs:")
display(
    complete_ball.groupby(
        ['extras_type', 'extras_2_runs'],
        dropna=False
    ).size().reset_index(name='count')
)


All non-zero extras_2_runs records:


,match_id,innings,runs,extras_type,extras1_runs,extras_2_runs
1485,13523,1,0,Overstep,1,4
2588,13614,1,0,other,5,4
7607,14112,1,0,Overstep,1,4
11386,14127,2,0,other,5,4
18844,14159,2,0,other,5,1
24453,15796,2,0,Overstep,1,1
28010,16424,2,0,Overstep,1,4
36854,17555,1,0,NaN,6,1
47440,20186,2,0,Overstep,1,1
47803,20188,1,0,other,5,1



Non-zero extras_2_runs distribution:
extras_2_runs
0    192430
1        19
4        16
Name: count, dtype: int64

Extras type and extras_2_runs:


,extras_type,extras_2_runs,count
0,Overstep,0,537
1,Overstep,1,12
2,Overstep,4,9
3,Penalty,0,1
4,byes,0,505
5,legbyes,0,2639
6,other,0,243
7,other,1,6
8,other,4,7
9,wide,0,7427


In [29]:
import os

print(os.listdir('/content'))

['.config', 'matches.csv', 'ball_by_ball.csv', 'teams.csv', 'competitions.csv', 'playing_eleven.csv', 'competition_team_player_mapping.csv', 'sample_data']


In [30]:

import os
import pandas as pd

files = [
    'matches.csv',
    'ball_by_ball.csv',
    'teams.csv',
    'competitions.csv',
    'playing_eleven.csv',
    'competition_team_player_mapping.csv'
]

for file in files:
    print(f"\n{'='*50}")
    print("FILE:", file)
    print("SIZE:", round(os.path.getsize(file) / 1024, 2), "KB")

    temp = pd.read_csv(file, nrows=3)

    print("COLUMNS:", temp.columns.tolist())
    display(temp.head(2))



FILE: matches.csv
SIZE: 120.96 KB
COLUMNS: ['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
0,13511,1331,1,2022-01-22,League,54,West Indies,28,West Indies,22,...,NaN,28,Fielding,0,NaN,NaN,NaN,0,28,West Indies Won by 9 Wickets
1,13512,1331,2,2022-01-23,League,54,West Indies,28,West Indies,22,...,478.0,28,Fielding,0,NaN,NaN,NaN,0,22,England Won by 1 Run



FILE: ball_by_ball.csv
SIZE: 40108.99 KB
COLUMNS: ['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,...,NaN,6,0,Pace,0,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC



FILE: teams.csv
SIZE: 6.76 KB
COLUMNS: ['comp_id', 'team_id', 'team_name']


,comp_id,team_id,team_name
0,1331,22,England
1,1331,28,West Indies


... (2 outputs left out)


FILE: playing_eleven.csv
SIZE: 943.64 KB
COLUMNS: ['match_id', 'team_id', 'player_id', 'batting_order', 'is_played', 'is_captain', 'is_wk', 'is_impact', 'is_impact_off']


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
0,13511,22,1187,1,True,False,False,0,0
1,13511,22,6273,2,True,False,False,0,0



FILE: competition_team_player_mapping.csv
SIZE: 452.12 KB
COLUMNS: ['comp_id', 'comp_name', 'team_id', 'team_name', 'player_id', 'player_name']


,comp_id,comp_name,team_id,team_name,player_id,player_name
0,1331,England in West Indies T20I Series 2022,22,England,478,Eoin Morgan
1,1331,England in West Indies T20I Series 2022,22,England,479,Adil Rashid


In [31]:
print("BATTING FEATURES")
print([
    'striker_id',
    'striker_name',
    'batting_order',
    'runs',
    'is_four',
    'is_six',
    'is_wicket',
    'overid',
    'over_ball_num'
])

print("\nBOWLING FEATURES")
print([
    'bowler_id',
    'bowler_name',
    'runs',
    'is_wicket',
    'dismissal_type',
    'extras_type',
    'overid',
    'over_ball_num'
])

BATTING FEATURES
['striker_id', 'striker_name', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'overid', 'over_ball_num']

BOWLING FEATURES
['bowler_id', 'bowler_name', 'runs', 'is_wicket', 'dismissal_type', 'extras_type', 'overid', 'over_ball_num']


In [32]:
print("COLUMN DATA TYPES")
print(complete_ball[
    ['runs', 'is_four', 'is_six', 'is_wicket',
     'batting_order', 'overid']
].dtypes)

print("\nRUNS DISTRIBUTION")
print(complete_ball['runs'].value_counts().sort_index())

print("\nFOUR DISTRIBUTION")
print(complete_ball['is_four'].value_counts(dropna=False))

print("\nSIX DISTRIBUTION")
print(complete_ball['is_six'].value_counts(dropna=False))

print("\nWICKET DISTRIBUTION")
print(complete_ball['is_wicket'].value_counts(dropna=False))

print("\nWICKET DISMISSAL TYPES")
print(
    complete_ball[complete_ball['is_wicket'] == 1]
    ['dismissal_type']
    .value_counts(dropna=False)
)

COLUMN DATA TYPES
runs             int64
is_four           bool
is_six            bool
is_wicket         bool
batting_order    int64
overid           int64
dtype: object

RUNS DISTRIBUTION
runs
0    75863
1    69069
2    12053
3      626
4    22521
5       22
6    12311
Name: count, dtype: int64

FOUR DISTRIBUTION
is_four
False    169964
True      22501
Name: count, dtype: int64

SIX DISTRIBUTION
is_six
False    180157
True      12308
Name: count, dtype: int64

WICKET DISTRIBUTION
is_wicket
False    182064
True      10401
Name: count, dtype: int64

WICKET DISMISSAL TYPES
dismissal_type
Caught                   7272
Bowled                   1676
LBW                       663
RunOut                    567
Stump                     198
Hit Wicket                 17
Retire Out                  6
Obstracting the field       1
Obstructing the field       1
Name: count, dtype: int64


In [33]:
print("FOUR FLAG MISMATCH")
print(
    complete_ball[
        (complete_ball['is_four'] == True) &
        (complete_ball['runs'] != 4)
    ][['runs', 'is_four', 'is_six', 'extras_type']]
)

print("\nSIX FLAG MISMATCH")
print(
    complete_ball[
        (complete_ball['is_six'] == True) &
        (complete_ball['runs'] != 6)
    ][['runs', 'is_four', 'is_six', 'extras_type']]
)

print("\nBOUNDARY COUNTS")
print("Four flags:", complete_ball['is_four'].sum())
print("Six flags:", complete_ball['is_six'].sum())

print("\nWIDE DELIVERIES")
print((complete_ball['extras_type'] == 'wide').sum())

FOUR FLAG MISMATCH
Empty DataFrame
Columns: [runs, is_four, is_six, extras_type]
Index: []

SIX FLAG MISMATCH
Empty DataFrame
Columns: [runs, is_four, is_six, extras_type]
Index: []

BOUNDARY COUNTS
Four flags: 22501
Six flags: 12308

WIDE DELIVERIES
7427


In [34]:
print(
    complete_ball
    .groupby('extras_type', dropna=False)
    .agg(
        deliveries=('match_id', 'size'),
        batter_runs=('runs', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum')
    )
    .reset_index()
    .to_string(index=False)
)

extras_type  deliveries  batter_runs  fours  sixes
   Overstep         558          824     79     45
    Penalty           1            0      0      0
       byes         505            0      0      0
    legbyes        2639            0      0      0
      other         256          403     40     26
       wide        7427            0      0      0
        NaN      181079       257886  22382  12237


In [35]:
for extra in ['Overstep', 'other', 'Penalty']:

    temp = complete_ball[
        complete_ball['extras_type'] == extra
    ]

    print(f"\n--- {extra} ---")

    print("Runs distribution:")
    print(temp['runs'].value_counts().sort_index())

    print("\nExtras_2 distribution:")
    print(temp['extras_2_runs'].value_counts(dropna=False))

    print("\nDismissal types:")
    print(temp['dismissal_type'].value_counts(dropna=False))

    print("\nSample records:")
    print(
        temp[['match_id', 'innings', 'overid',
              'over_ball_num', 'runs', 'is_wicket',
              'dismissal_type', 'extras_2_runs']]
        .head(5).to_string(index=False)
    )


--- Overstep ---
Runs distribution:
runs
0    233
1    166
2     33
3      2
4     79
6     45
Name: count, dtype: int64

Extras_2 distribution:
extras_2_runs
0    537
1     12
4      9
Name: count, dtype: int64

Dismissal types:
dismissal_type
NaN       553
RunOut      5
Name: count, dtype: int64

Sample records:
match_id  innings  overid  over_ball_num  runs  is_wicket dismissal_type  extras_2_runs
   13513        1       4              2     0      False            NaN              0
   13515        2      20              1     1      False            NaN              0
   13523        1       1              4     0      False            NaN              4
   13523        1       1              5     1      False            NaN              0
   14094        1       1              1     0      False            NaN              0

--- other ---
Runs distribution:
runs
0    123
1     48
2     18
3      1
4     40
6     26
Name: count, dtype: int64

Extras_2 distribution:
extras_2_run

In [36]:
for extra in ['Overstep', 'other', 'wide', 'NaN']:

    if extra == 'NaN':
        temp = complete_ball[complete_ball['extras_type'].isna()]
    else:
        temp = complete_ball[
            complete_ball['extras_type'] == extra
        ]

    print(f"\n--- {extra} ---")

    print("Over ball number distribution:")
    print(temp['over_ball_num'].value_counts().sort_index())

    print("\nSample over sequences:")
    sample_match = temp['match_id'].iloc[0]

    print(
        complete_ball[
            (complete_ball['match_id'] == sample_match) &
            (complete_ball['innings'] == temp['innings'].iloc[0]) &
            (complete_ball['overid'] == temp['overid'].iloc[0])
        ][['overid', 'over_ball_num', 'runs', 'extras_type']]
        .to_string(index=False)
    )


--- Overstep ---
Over ball number distribution:
over_ball_num
1     76
2     99
3    106
4     82
5     68
6    105
7     15
8      4
9      3
Name: count, dtype: int64

Sample over sequences:
 overid  over_ball_num  runs extras_type
      4              1     2         NaN
      4              2     0    Overstep
      4              3     0         NaN
      4              4     6         NaN
      4              5     4         NaN
      4              6     6         NaN
      4              7     0         NaN

--- other ---
Over ball number distribution:
over_ball_num
1    43
2    34
3    31
4    42
5    46
6    41
7    13
8     5
9     1
Name: count, dtype: int64

Sample over sequences:
 overid  over_ball_num  runs extras_type
     10              1     0       other
     10              2     6         NaN
     10              3     1         NaN
     10              4     2         NaN
     10              5     1         NaN
     10              6     0         NaN
     10  

In [37]:
batting_summary = (
    complete_ball
    .groupby(['striker_id', 'striker_name'])
    .agg(
        total_runs=('runs', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum'),
        total_deliveries=('match_id', 'size'),
        innings=('match_id', 'nunique')
    )
    .reset_index()
)

batting_summary['boundary_runs'] = (
    batting_summary['fours'] * 4 +
    batting_summary['sixes'] * 6
)

batting_summary['non_boundary_runs'] = (
    batting_summary['total_runs'] -
    batting_summary['boundary_runs']
)

batting_summary = batting_summary.sort_values(
    'total_runs', ascending=False
)

print("Total batters:", len(batting_summary))
print(batting_summary.head(15).to_string(index=False))

Total batters: 824
 striker_id         striker_name  total_runs  fours  sixes  total_deliveries  innings  boundary_runs  non_boundary_runs
       1699     Suryakumar Yadav        4549    428    246              2822      138           3188               1361
        813          Jos Buttler        4367    429    203              2894      120           2934               1433
       6281      Abhishek Sharma        4091    393    269              2314      126           3186                905
       6288         Shubman Gill        3984    397    141              2768      110           2434               1550
         47          Virat Kohli        3567    344    124              2545       92           2120               1447
       4117         Ishan Kishan        3484    365    153              2484      119           2378               1106
       7897 Namboori Tilak Varma        3459    266    167              2508      121           2066               1393
       2258         S

In [38]:
# Har batter ki unique match + innings combination
batter_innings = (
    complete_ball[
        ['striker_id', 'match_id', 'innings']
    ]
    .drop_duplicates()
    .groupby('striker_id')
    .size()
    .reset_index(name='innings')
)

# Purane innings column ko replace karna
batting_summary = (
    batting_summary
    .drop(columns=['innings'])
    .merge(batter_innings, on='striker_id', how='left')
)

batting_summary = batting_summary.sort_values(
    'total_runs', ascending=False
)

print(
    batting_summary.head(15).to_string(index=False)
)

 striker_id         striker_name  total_runs  fours  sixes  total_deliveries  boundary_runs  non_boundary_runs  innings
       1699     Suryakumar Yadav        4549    428    246              2822           3188               1361      138
        813          Jos Buttler        4367    429    203              2894           2934               1433      120
       6281      Abhishek Sharma        4091    393    269              2314           3186                905      126
       6288         Shubman Gill        3984    397    141              2768           2434               1550      110
         47          Virat Kohli        3567    344    124              2545           2120               1447       92
       4117         Ishan Kishan        3484    365    153              2484           2378               1106      119
       7897 Namboori Tilak Varma        3459    266    167              2508           2066               1393      121
       2258         Sanju Samson        

In [39]:
# Har batter ki har innings ka performance
temp = complete_ball.copy()

# Sirf us batter ka dismissal count hoga jo actually dismissed hua
temp['batter_out'] = (
    temp['is_wicket'] &
    (temp['dismissed_player_id'] == temp['striker_id'])
)

batter_innings_summary = (
    temp
    .groupby([
        'match_id',
        'innings',
        'striker_id',
        'striker_name',
        'batting_order'
    ])
    .agg(
        runs=('runs', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum'),
        deliveries=('match_id', 'size'),
        dismissed=('batter_out', 'sum')
    )
    .reset_index()
)

print("Total batter innings:", len(batter_innings_summary))

print("\nSample records:")
print(
    batter_innings_summary.head(15)
    .to_string(index=False)
)

Total batter innings: 13122

Sample records:
match_id  innings  striker_id    striker_name  batting_order  runs  fours  sixes  deliveries  dismissed
   13511        1         478     Eoin Morgan              5    17      0      1          32          1
   13511        1         479     Adil Rashid              9    22      3      0          18          1
   13511        1         722     Liam Dawson              7     2      0      0           9          1
   13511        1         786    Chris Jordan              8    28      0      3          23          1
   13511        1        1187       Jason Roy              1     6      0      1           6          1
   13511        1        1210     James Vince              3    14      2      1          12          1
   13511        1        1596       Moeen Ali              4     0      0      0           1          1
   13511        1        1924    Sam Billings              6     2      0      0           4          1
   13511        1  

In [40]:
print("DATA TYPES")
print(complete_ball[
    ['innings_ball_num', 'is_wicket', 'striker_id']
].dtypes)

print("\nSAMPLE BALL SEQUENCE")
print(
    complete_ball[
        (complete_ball['match_id'] == 13511) &
        (complete_ball['innings'] == 1)
    ][[
        'innings_ball_num',
        'overid',
        'over_ball_num',
        'striker_name',
        'runs',
        'is_wicket',
        'dismissal_type'
    ]].head(25).to_string(index=False)
)

print("\nDUPLICATE BALL NUMBERS")
print(
    complete_ball.duplicated(
        ['match_id', 'innings', 'innings_ball_num']
    ).sum()
)

print("\nMISSING BALL NUMBERS")
print(complete_ball['innings_ball_num'].isna().sum())

DATA TYPES
innings_ball_num    int64
is_wicket            bool
striker_id          int64
dtype: object

SAMPLE BALL SEQUENCE
Empty DataFrame
Columns: [innings_ball_num, overid, over_ball_num, striker_name, runs, is_wicket, dismissal_type]
Index: []

DUPLICATE BALL NUMBERS
0

MISSING BALL NUMBERS
0


In [41]:
print("MATCH ID DATA TYPE:")
print(complete_ball['match_id'].dtype)

print("\nMATCH ID SAMPLE:")
print(complete_ball['match_id'].head(10).tolist())

print("\nMATCH 13511 CHECK:")
print(
    complete_ball[
        complete_ball['match_id'].astype(str).str.strip() == '13511'
    ][[
        'match_id', 'innings', 'innings_ball_num',
        'striker_name', 'runs', 'is_wicket'
    ]].head(25).to_string(index=False)
)

MATCH ID DATA TYPE:
object

MATCH ID SAMPLE:
['13511', '13511', '13511', '13511', '13511', '13511', '13511', '13511', '13511', '13511']

MATCH 13511 CHECK:
match_id  innings  innings_ball_num striker_name  runs  is_wicket
   13511        1                 1    Jason Roy     0      False
   13511        1                 2    Jason Roy     0      False
   13511        1                 3    Jason Roy     0      False
   13511        1                 4    Jason Roy     0      False
   13511        1                 5    Jason Roy     6      False
   13511        1                 6    Jason Roy     0       True
   13511        1                 7   Tom Banton     0      False
   13511        1                 8   Tom Banton     0      False
   13511        1                 9   Tom Banton     0      False
   13511        1                10   Tom Banton     4      False
   13511        1                11   Tom Banton     0       True
   13511        1                12    Moeen Ali    

In [42]:
# Har batter ki first faced delivery
first_ball = (
    complete_ball
    .groupby(
        ['match_id', 'innings', 'striker_id']
    )['innings_ball_num']
    .min()
    .reset_index(name='first_ball_faced')
)

# Har innings mein delivery sequence par wickets ka cumulative count
wicket_sequence = complete_ball[
    ['match_id', 'innings', 'innings_ball_num', 'is_wicket']
].copy()

wicket_sequence = wicket_sequence.sort_values(
    ['match_id', 'innings', 'innings_ball_num']
)

wicket_sequence['wickets_before'] = (
    wicket_sequence
    .groupby(['match_id', 'innings'])['is_wicket']
    .cumsum()
    - wicket_sequence['is_wicket'].astype(int)
)

# First faced ball par wickets se pehle ka count
first_ball = first_ball.merge(
    wicket_sequence[
        ['match_id', 'innings', 'innings_ball_num', 'wickets_before']
    ],
    left_on=['match_id', 'innings', 'first_ball_faced'],
    right_on=['match_id', 'innings', 'innings_ball_num'],
    how='left'
)

first_ball = first_ball.drop(columns='innings_ball_num')

# Batter innings summary mein feature add karo
batter_innings_summary = batter_innings_summary.merge(
    first_ball[
        ['match_id', 'innings', 'striker_id', 'wickets_before']
    ],
    on=['match_id', 'innings', 'striker_id'],
    how='left'
)

print("WICKETS BEFORE FIRST FACED BALL")
print(
    batter_innings_summary['wickets_before']
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSAMPLE - MATCH 13511")
print(
    batter_innings_summary[
        batter_innings_summary['match_id'] == '13511'
    ][[
        'striker_name', 'batting_order',
        'runs', 'first_ball_faced' if 'first_ball_faced' in batter_innings_summary.columns else 'wickets_before',
        'wickets_before'
    ]].to_string(index=False)
)

WICKETS BEFORE FIRST FACED BALL
wickets_before
0    3046
1    1787
2    1619
3    1519
4    1376
5    1182
6     974
7     722
8     554
9     343
Name: count, dtype: int64

SAMPLE - MATCH 13511
   striker_name  batting_order  runs  wickets_before  wickets_before
    Eoin Morgan              5    17               3               3
    Adil Rashid              9    22               7               7
    Liam Dawson              7     2               5               5
   Chris Jordan              8    28               6               6
      Jason Roy              1     6               0               0
    James Vince              3    14               3               3
      Moeen Ali              4     0               2               2
   Sam Billings              6     2               4               4
  Saqib Mahmood             10     5               8               8
     Tom Banton              2     4               1               1
Nicholas Pooran              3    27          

In [43]:
# Batter ka first appearance: striker ya non-striker
striker_first = complete_ball[
    ['match_id', 'innings', 'striker_id', 'innings_ball_num']
].rename(columns={'striker_id': 'player_id'})

non_striker_first = complete_ball[
    ['match_id', 'innings', 'non_striker_id', 'innings_ball_num']
].rename(columns={'non_striker_id': 'player_id'})

all_appearances = pd.concat(
    [striker_first, non_striker_first],
    ignore_index=True
)

first_appearance = (
    all_appearances
    .groupby(['match_id', 'innings', 'player_id'])
    ['innings_ball_num']
    .min()
    .reset_index(name='first_appearance_ball')
)

comparison = first_ball.merge(
    first_appearance,
    left_on=['match_id', 'innings', 'striker_id'],
    right_on=['match_id', 'innings', 'player_id'],
    how='left'
)

comparison['ball_difference'] = (
    comparison['first_ball_faced'] -
    comparison['first_appearance_ball']
)

print("FIRST APPEARANCE vs FIRST BALL FACED")
print(comparison['ball_difference'].value_counts().sort_index())

print("\nDelayed first faced ball:")
print((comparison['ball_difference'] > 0).sum())

print("\nMaximum delay:")
print(comparison['ball_difference'].max())

FIRST APPEARANCE vs FIRST BALL FACED
ball_difference
0     9743
1     1180
2      648
3      437
4      300
5      218
6      365
7      112
8       37
9       29
10       9
11       7
12      17
13      13
14       1
15       2
16       1
18       1
19       1
20       1
Name: count, dtype: int64

Delayed first faced ball:
3379

Maximum delay:
20


In [44]:
# First appearance ko delivery sequence ke saath merge karo
appearance_pressure = first_appearance.merge(
    wicket_sequence[
        ['match_id', 'innings', 'innings_ball_num', 'wickets_before']
    ],
    left_on=[
        'match_id', 'innings', 'first_appearance_ball'
    ],
    right_on=[
        'match_id', 'innings', 'innings_ball_num'
    ],
    how='left'
)

appearance_pressure = appearance_pressure.drop(
    columns=['innings_ball_num']
)

appearance_pressure = appearance_pressure.rename(
    columns={'player_id': 'striker_id'}
)

# Batter innings summary mein correct pressure feature add karo
batter_innings_summary = batter_innings_summary.drop(
    columns=['wickets_before']
)

batter_innings_summary = batter_innings_summary.merge(
    appearance_pressure[
        ['match_id', 'innings', 'striker_id', 'wickets_before']
    ],
    on=['match_id', 'innings', 'striker_id'],
    how='left'
)

print("FIRST APPEARANCE WICKETS DISTRIBUTION")

print(
    batter_innings_summary['wickets_before']
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMISSING VALUES:")
print(batter_innings_summary['wickets_before'].isna().sum())

FIRST APPEARANCE WICKETS DISTRIBUTION
wickets_before
0    3261
1    1627
2    1595
3    1517
4    1370
5    1194
6     961
7     729
8     544
9     324
Name: count, dtype: int64

MISSING VALUES:
0


In [45]:
pressure_performance = (
    batter_innings_summary
    .groupby('wickets_before')
    .agg(
        batter_innings=('striker_id', 'size'),
        avg_runs=('runs', 'mean'),
        median_runs=('runs', 'median'),
        total_runs=('runs', 'sum'),
        scores_30_plus=('runs', lambda x: (x >= 30).sum()),
        scores_50_plus=('runs', lambda x: (x >= 50).sum()),
        dismissals=('dismissed', 'sum'),
        fours=('fours', 'sum'),
        sixes=('sixes', 'sum')
    )
    .reset_index()
)

pressure_performance['dismissal_pct'] = (
    pressure_performance['dismissals'] /
    pressure_performance['batter_innings'] * 100
).round(2)

pressure_performance['30_plus_pct'] = (
    pressure_performance['scores_30_plus'] /
    pressure_performance['batter_innings'] * 100
).round(2)

print(pressure_performance.round(2).to_string(index=False))

 wickets_before  batter_innings  avg_runs  median_runs  total_runs  scores_30_plus  scores_50_plus  dismissals  fours  sixes  dismissal_pct  30_plus_pct
              0            3261     28.11         20.0       91661            1226             673        3052   9348   4114          93.59        37.60
              1            1627     25.42         18.0       41353             553             281        1452   3725   1940          89.24        33.99
              2            1595     24.39         19.0       38909             531             235        1320   3113   1844          82.76        33.29
              3            1517     20.34         15.0       30859             401             128        1159   2173   1588          76.40        26.43
              4            1370     16.72         12.0       22907             261              67         986   1667   1164          71.97        19.05
              5            1194     14.00         10.0       16717             150

In [46]:
position_analysis = (
    batter_innings_summary
    .groupby('batting_order')
    .agg(
        innings=('striker_id', 'size'),
        avg_runs=('runs', 'mean'),
        median_runs=('runs', 'median'),
        avg_wickets_before=('wickets_before', 'mean'),
        scores_30_plus=('runs', lambda x: (x >= 30).sum()),
        scores_50_plus=('runs', lambda x: (x >= 50).sum())
    )
    .reset_index()
)

position_analysis['30_plus_pct'] = (
    position_analysis['scores_30_plus'] /
    position_analysis['innings'] * 100
).round(2)

print(position_analysis.round(2).to_string(index=False))

 batting_order  innings  avg_runs  median_runs  avg_wickets_before  scores_30_plus  scores_50_plus  30_plus_pct
             1     1630     27.79         19.0                0.00             619             329        37.98
             2     1626     28.47         20.0                0.00             606             344        37.27
             3     1621     25.35         18.0                1.00             549             278        33.87
             4     1594     24.46         19.0                1.99             532             238        33.38
             5     1516     20.35         16.0                2.99             402             126        26.52
             6     1370     16.78         12.0                3.99             262              69        19.12
             7     1201     14.03         10.0                4.99             152              30        12.66
             8      962      9.45          6.0                5.99              64              10      

In [47]:
# First appearance delivery ka over identify karo
entry_over = first_appearance.merge(
    complete_ball[
        ['match_id', 'innings', 'innings_ball_num', 'overid']
    ],
    left_on=[
        'match_id', 'innings', 'first_appearance_ball'
    ],
    right_on=[
        'match_id', 'innings', 'innings_ball_num'
    ],
    how='left'
)

entry_over = entry_over.rename(
    columns={'player_id': 'striker_id',
             'overid': 'entry_over'}
)

# Batter innings summary mein add karo
batter_innings_summary = batter_innings_summary.drop(
    columns=['entry_over'],
    errors='ignore'
)

batter_innings_summary = batter_innings_summary.merge(
    entry_over[
        ['match_id', 'innings', 'striker_id', 'entry_over']
    ],
    on=['match_id', 'innings', 'striker_id'],
    how='left'
)

# Position-wise entry over analysis
entry_analysis = (
    batter_innings_summary
    .groupby('batting_order')
    .agg(
        innings=('striker_id', 'size'),
        avg_entry_over=('entry_over', 'mean'),
        median_entry_over=('entry_over', 'median'),
        earliest_entry=('entry_over', 'min'),
        latest_entry=('entry_over', 'max')
    )
    .reset_index()
)

print(entry_analysis.round(2).to_string(index=False))

print("\nMissing entry overs:")
print(batter_innings_summary['entry_over'].isna().sum())

 batting_order  innings  avg_entry_over  median_entry_over  earliest_entry  latest_entry
             1     1630            1.00                1.0               1             1
             2     1626            1.00                1.0               1             1
             3     1621            4.05                3.0               1            18
             4     1594            7.31                6.0               1            20
             5     1516           10.16               10.0               2            20
             6     1370           12.56               13.0               2            20
             7     1201           14.64               15.0               3            20
             8      962           16.14               17.0               4            20
             9      731           17.01               18.0               6            20
            10      542           17.66               18.0               7            20
            11      3

In [48]:
import numpy as np

batter_innings_summary['entry_phase'] = np.select(
    [
        batter_innings_summary['entry_over'].between(1, 6),
        batter_innings_summary['entry_over'].between(7, 15),
        batter_innings_summary['entry_over'].between(16, 20)
    ],
    [
        'Powerplay',
        'Middle',
        'Death'
    ],
    default='Unknown'
)

phase_analysis = (
    batter_innings_summary
    .groupby(['batting_order', 'entry_phase'])
    .agg(
        innings=('striker_id', 'size'),
        avg_runs=('runs', 'mean'),
        median_runs=('runs', 'median'),
        avg_wickets_before=('wickets_before', 'mean'),
        scores_30_plus=('runs', lambda x: (x >= 30).sum())
    )
    .reset_index()
)

phase_analysis['30_plus_pct'] = (
    phase_analysis['scores_30_plus'] /
    phase_analysis['innings'] * 100
).round(2)

print(phase_analysis.round(2).to_string(index=False))

 batting_order entry_phase  innings  avg_runs  median_runs  avg_wickets_before  scores_30_plus  30_plus_pct
             1   Powerplay     1630     27.79         19.0                0.00             619        37.98
             2   Powerplay     1626     28.47         20.0                0.00             606        37.27
             3       Death        9     17.22         15.0                1.00               2        22.22
             3      Middle      274     22.95         17.0                1.00              86        31.39
             3   Powerplay     1338     25.90         19.0                1.00             461        34.45
             4       Death       63     10.56          7.0                1.98               3         4.76
             4      Middle      716     22.90         19.0                1.99             227        31.70
             4   Powerplay      815     26.89         20.0                2.00             302        37.06
             5       Death  

In [49]:
# 1. Position + entry phase ka baseline
phase_baseline = (
    batter_innings_summary
    .groupby(['batting_order', 'entry_phase'])
    .agg(
        expected_runs=('runs', 'mean'),
        sample_size=('runs', 'size')
    )
    .reset_index()
)

# 2. Batting order ka overall baseline
order_baseline = (
    batter_innings_summary
    .groupby('batting_order')['runs']
    .mean()
    .to_dict()
)

# 3. Baseline ko batter innings data mein add karo
batter_innings_summary = batter_innings_summary.merge(
    phase_baseline,
    on=['batting_order', 'entry_phase'],
    how='left',
    suffixes=('', '_phase')
)

# 4. Small samples ke liye fallback
batter_innings_summary['expected_runs'] = np.where(
    batter_innings_summary['sample_size'] >= 30,
    batter_innings_summary['expected_runs'],
    batter_innings_summary['batting_order'].map(order_baseline)
)

# 5. Expected runs se kitna better/worse perform kiya
batter_innings_summary['runs_vs_expected'] = (
    batter_innings_summary['runs'] -
    batter_innings_summary['expected_runs']
)

print("BASELINE CHECK:")
print(
    phase_baseline.round(2).to_string(index=False)
)

print("\nSAMPLE COMPARISON:")
print(
    batter_innings_summary[
        ['striker_name', 'batting_order', 'entry_phase',
         'runs', 'expected_runs', 'runs_vs_expected']
    ].head(15).round(2).to_string(index=False)
)

BASELINE CHECK:
 batting_order entry_phase  expected_runs  sample_size
             1   Powerplay          27.79         1630
             2   Powerplay          28.47         1626
             3       Death          17.22            9
             3      Middle          22.95          274
             3   Powerplay          25.90         1338
             4       Death          10.56           63
             4      Middle          22.90          716
             4   Powerplay          26.89          815
             5       Death          10.59          188
             5      Middle          20.50          950
             5   Powerplay          24.84          378
             6       Death          10.03          361
             6      Middle          18.92          900
             6   Powerplay          21.49          109
             7       Death           9.24          548
             7      Middle          17.80          630
             7   Powerplay          24.78        

In [52]:

# Fresh copy to avoid duplicate columns from previous runs
batter_eval = batter_innings_summary.copy()

group_cols = ['batting_order', 'entry_phase']

# Remove old baseline columns if they exist
old_cols = [
    'group_runs', 'sample_size',
    'expected_runs_loo', 'runs_vs_expected'
]

batter_eval = batter_eval.drop(
    columns=[c for c in old_cols if c in batter_eval.columns]
)

# Calculate group baseline
baseline = (
    batter_eval.groupby(group_cols)['runs']
    .agg(['sum', 'count'])
    .reset_index()
    .rename(columns={
        'sum': 'group_runs',
        'count': 'sample_size'
    })
)

# Merge baseline
batter_eval = batter_eval.merge(
    baseline,
    on=group_cols,
    how='left'
)

# Leave-One-Out expected runs
batter_eval['expected_runs_loo'] = np.where(
    batter_eval['sample_size'] > 1,
    (batter_eval['group_runs'] - batter_eval['runs']) /
    (batter_eval['sample_size'] - 1),
    np.nan
)

# Minimum 30 innings
batter_eval['runs_vs_expected'] = np.where(
    batter_eval['sample_size'] >= 30,
    batter_eval['runs'] - batter_eval['expected_runs_loo'],
    np.nan
)

print("Columns:", batter_eval.columns.tolist())

print("\nSAMPLE COMPARISON:")
print(
    batter_eval[
        ['striker_name', 'batting_order', 'entry_phase',
         'runs', 'expected_runs_loo', 'runs_vs_expected']
    ].head(15).round(2).to_string(index=False)
)

print("\nEligible innings:",
      batter_eval['runs_vs_expected'].notna().sum())


Columns: ['match_id', 'innings', 'striker_id', 'striker_name', 'batting_order', 'runs', 'fours', 'sixes', 'deliveries', 'dismissed', 'wickets_before', 'entry_over', 'entry_phase', 'expected_runs', 'sample_size_x', 'group_runs_x', 'sample_size_y', 'group_runs_y', 'group_runs', 'sample_size', 'expected_runs_loo', 'runs_vs_expected']

SAMPLE COMPARISON:
   striker_name  batting_order entry_phase  runs  expected_runs_loo  runs_vs_expected
    Eoin Morgan              5   Powerplay    17              24.86             -7.86
    Adil Rashid              9      Middle    22               9.57             12.43
    Liam Dawson              7      Middle     2              17.82            -15.82
   Chris Jordan              8      Middle    28              13.86             14.14
      Jason Roy              1   Powerplay     6              27.80            -21.80
    James Vince              3   Powerplay    14              25.91            -11.91
      Moeen Ali              4   Powerplay   

In [53]:

player_performance = (
    batter_eval[batter_eval['runs_vs_expected'].notna()]
    .groupby(['striker_id', 'striker_name'])
    .agg(
        eligible_innings=('runs_vs_expected', 'count'),
        total_runs=('runs', 'sum'),
        avg_runs=('runs', 'mean'),
        avg_above_expected=('runs_vs_expected', 'mean'),
        median_above_expected=('runs_vs_expected', 'median'),
        positive_innings=('runs_vs_expected', lambda x: (x > 0).sum()),
        consistency_std=('runs_vs_expected', 'std')
    )
    .reset_index()
)

player_performance['positive_pct'] = (
    player_performance['positive_innings'] /
    player_performance['eligible_innings'] * 100
).round(2)

# Initial minimum sample filter
qualified_players = player_performance[
    player_performance['eligible_innings'] >= 10
].copy()

qualified_players = qualified_players.sort_values(
    ['avg_above_expected', 'positive_pct'],
    ascending=False
)

print("Total qualified players:", len(qualified_players))

print("\nTOP 20 — AVERAGE PERFORMANCE ABOVE EXPECTATION:")
print(
    qualified_players[
        ['striker_name', 'eligible_innings', 'total_runs',
         'avg_runs', 'avg_above_expected',
         'median_above_expected', 'positive_pct',
         'consistency_std']
    ].head(20).round(2).to_string(index=False)
)

print("\nBOTTOM 10:")
print(
    qualified_players[
        ['striker_name', 'eligible_innings',
         'avg_above_expected', 'positive_pct']
    ].tail(10).round(2).to_string(index=False)
)


Total qualified players: 326

TOP 20 — AVERAGE PERFORMANCE ABOVE EXPECTATION:
        striker_name  eligible_innings  total_runs  avg_runs  avg_above_expected  median_above_expected  positive_pct  consistency_std
       Sai Sudharsan                57        2515     44.12               17.34                  19.12         66.67            28.38
Vaibhav Sooryavanshi                31        1270     40.97               12.67                  10.53         61.29            33.18
     Mohammad Rizwan                37        1467     39.65               11.88                   6.21         51.35            31.39
         Virat Kohli                92        3567     38.77               11.39                   7.16         56.52            30.02
        Lokesh Rahul                73        2837     38.86               11.35                   4.53         57.53            32.12
    Heinrich Klaasen                72        2363     32.82               11.25                   6.34         

In [54]:

# Compare different minimum innings thresholds
for minimum in [10, 20, 30, 50]:

    test = player_performance[
        player_performance['eligible_innings'] >= minimum
    ].copy()

    test = test.sort_values(
        'avg_above_expected',
        ascending=False
    )

    print(f"\n{'='*65}")
    print(f"MINIMUM INNINGS: {minimum}")
    print(f"QUALIFIED PLAYERS: {len(test)}")
    print("="*65)

    print(
        test[
            ['striker_name', 'eligible_innings',
             'avg_above_expected', 'positive_pct',
             'consistency_std']
        ].head(15).round(2).to_string(index=False)
    )



MINIMUM INNINGS: 10
QUALIFIED PLAYERS: 326
        striker_name  eligible_innings  avg_above_expected  positive_pct  consistency_std
       Sai Sudharsan                57               17.34         66.67            28.38
Vaibhav Sooryavanshi                31               12.67         61.29            33.18
     Mohammad Rizwan                37               11.88         51.35            31.39
         Virat Kohli                92               11.39         56.52            30.02
        Lokesh Rahul                73               11.35         57.53            32.12
    Heinrich Klaasen                72               11.25         62.50            23.26
         Jos Buttler               120                9.56         55.00            31.03
     Ashutosh Sharma                25                8.53         72.00            16.55
    Suryakumar Yadav               138                8.27         55.07            28.19
        David Warner                45                7.

In [57]:


# 1. Pressure category-wise innings distribution
print("PRESSURE CATEGORY SUMMARY:")

print(
    batter_eval.groupby('pressure_level')
    .agg(
        innings=('runs', 'size'),
        eligible_innings=('runs_vs_expected', 'count'),
        avg_runs=('runs', 'mean'),
        avg_wickets_before=('wickets_before', 'mean')
    ).round(2)
)

# 2. Player-wise pressure category sample count
print("\nPLAYER PRESSURE SAMPLE DISTRIBUTION:")

print(
    pressure_analysis.pivot(
        index='striker_name',
        columns='pressure_level',
        values='innings'
    ).fillna(0).describe().round(2)
)

# 3. Top 15 players in medium + high pressure
high_pressure_players = (
    pressure_analysis[
        pressure_analysis['pressure_level'].isin(['Medium', 'High'])
    ]
    .groupby(['striker_id', 'striker_name'])
    .agg(
        pressure_innings=('innings', 'sum'),
        avg_above_expected=('avg_above_expected', 'mean'),
        positive_pct=('positive_pct', 'mean')
    )
    .reset_index()
)

high_pressure_players = high_pressure_players[
    high_pressure_players['pressure_innings'] >= 10
].sort_values('avg_above_expected', ascending=False)

print("\nTOP 15 MEDIUM + HIGH PRESSURE PLAYERS:")

print(
    high_pressure_players.head(15)
    .round(2).to_string(index=False)
)


PRESSURE CATEGORY SUMMARY:
                innings  eligible_innings  avg_runs  avg_wickets_before
pressure_level                                                         
High               2558              2553      6.53                7.09
Low                6483              6474     26.52                0.74
Medium             4081              4058     17.27                3.92

PLAYER PRESSURE SAMPLE DISTRIBUTION:


ValueError: Index contains duplicate entries, cannot reshape

In [58]:

# Count players and innings in each pressure category
pressure_distribution = (
    pressure_analysis
    .groupby('pressure_level')
    .agg(
        players=('striker_id', 'nunique'),
        total_innings=('innings', 'sum'),
        avg_innings_per_player=('innings', 'mean')
    )
    .round(2)
)

print(pressure_distribution)


                players  total_innings  avg_innings_per_player
pressure_level                                                
High                442           2553                    5.78
Low                 394           6474                   16.43
Medium              385           4058                   10.54


In [59]:

# Combine Medium + High pressure innings
high_pressure_players = (
    batter_eval[
        (batter_eval['pressure_level'].isin(['Medium', 'High'])) &
        (batter_eval['runs_vs_expected'].notna())
    ]
    .groupby(['striker_id', 'striker_name'])
    .agg(
        pressure_innings=('runs_vs_expected', 'count'),
        total_runs=('runs', 'sum'),
        avg_above_expected=('runs_vs_expected', 'mean'),
        median_above_expected=('runs_vs_expected', 'median'),
        positive_innings=('runs_vs_expected',
                          lambda x: (x > 0).sum())
    )
    .reset_index()
)

high_pressure_players['positive_pct'] = (
    high_pressure_players['positive_innings'] /
    high_pressure_players['pressure_innings'] * 100
).round(2)

# Minimum 10 pressure innings
high_pressure_players = high_pressure_players[
    high_pressure_players['pressure_innings'] >= 10
].sort_values('avg_above_expected', ascending=False)

print("TOP 20 MEDIUM + HIGH PRESSURE BATTERS")

print(
    high_pressure_players[
        ['striker_name', 'pressure_innings', 'total_runs',
         'avg_above_expected', 'median_above_expected',
         'positive_pct']
    ].head(20).round(2).to_string(index=False)
)


TOP 20 MEDIUM + HIGH PRESSURE BATTERS
        striker_name  pressure_innings  total_runs  avg_above_expected  median_above_expected  positive_pct
       Towhid Hridoy                10         321               10.88                   3.34         50.00
    Heinrich Klaasen                44        1323                9.92                   7.51         65.91
     Ashutosh Sharma                25         523                8.53                   6.21         72.00
     Nicholas Pooran                43        1184                7.81                  -1.50         48.84
        Matthew Wade                19         431                7.62                  10.22         73.68
         Rinku Singh                77        1822                6.40                   1.98         54.55
       Andre Russell                54        1177                6.14                   2.55         53.70
           Tim David                87        1926                6.13                   3.50     

In [60]:

# Separate Medium and High pressure performance
separate_pressure = (
    batter_eval[
        batter_eval['pressure_level'].isin(['Medium', 'High']) &
        batter_eval['runs_vs_expected'].notna()
    ]
    .groupby([
        'striker_id', 'striker_name', 'pressure_level'
    ])
    .agg(
        innings=('runs_vs_expected', 'count'),
        total_runs=('runs', 'sum'),
        avg_above_expected=('runs_vs_expected', 'mean'),
        median_above_expected=('runs_vs_expected', 'median'),
        positive_innings=('runs_vs_expected',
                          lambda x: (x > 0).sum())
    )
    .reset_index()
)

separate_pressure['positive_pct'] = (
    separate_pressure['positive_innings'] /
    separate_pressure['innings'] * 100
).round(2)

for level in ['High', 'Medium']:

    result = separate_pressure[
        (separate_pressure['pressure_level'] == level) &
        (separate_pressure['innings'] >= 10)
    ].sort_values(
        'avg_above_expected',
        ascending=False
    )

    print(f"\n{'='*65}")
    print(f"{level.upper()} PRESSURE — TOP 15 BATTERS")
    print(f"{'='*65}")

    print(
        result[
            ['striker_name', 'innings', 'total_runs',
             'avg_above_expected', 'median_above_expected',
             'positive_pct']
        ].head(15).round(2).to_string(index=False)
    )



HIGH PRESSURE — TOP 15 BATTERS
        striker_name  innings  total_runs  avg_above_expected  median_above_expected  positive_pct
    Romario Shepherd       25         462                8.23                   8.12         68.00
Mahendra Singh Dhoni       17         267                7.59                   5.02         64.71
       Jamie Overton       10         155                5.96                   4.57         60.00
        Corbin Bosch       11         164                5.60                   4.02         63.64
          Mark Adair       18         257                4.32                   1.52         55.56
       Deepak Chahar       11          94                2.65                  -3.66         45.45
       Faheem Ashraf       21         245                2.54                   0.10         57.14
         Rashid Khan       49         566                2.52                  -2.99         38.78
    Mitchell Santner       15         164                2.51                

In [61]:

for minimum in [10, 15, 20]:

    result = separate_pressure[
        (separate_pressure['pressure_level'] == 'High') &
        (separate_pressure['innings'] >= minimum)
    ].copy()

    result = result.sort_values(
        'avg_above_expected',
        ascending=False
    )

    print(f"\n{'='*65}")
    print(f"HIGH PRESSURE — MINIMUM {minimum} INNINGS")
    print(f"QUALIFIED PLAYERS: {len(result)}")
    print("="*65)

    print(
        result[
            ['striker_name', 'innings', 'total_runs',
             'avg_above_expected', 'median_above_expected',
             'positive_pct']
        ].head(15).round(2).to_string(index=False)
    )



HIGH PRESSURE — MINIMUM 10 INNINGS
QUALIFIED PLAYERS: 90
        striker_name  innings  total_runs  avg_above_expected  median_above_expected  positive_pct
    Romario Shepherd       25         462                8.23                   8.12         68.00
Mahendra Singh Dhoni       17         267                7.59                   5.02         64.71
       Jamie Overton       10         155                5.96                   4.57         60.00
        Corbin Bosch       11         164                5.60                   4.02         63.64
          Mark Adair       18         257                4.32                   1.52         55.56
       Deepak Chahar       11          94                2.65                  -3.66         45.45
       Faheem Ashraf       21         245                2.54                   0.10         57.14
         Rashid Khan       49         566                2.52                  -2.99         38.78
    Mitchell Santner       15         164          

In [62]:

top_players = [
    'Romario Shepherd',
    'Mahendra Singh Dhoni',
    'Mark Adair',
    'Faheem Ashraf',
    'Rashid Khan',
    'Mitchell Santner',
    'Patrick Cummins',
    'Akeal Hosein',
    'Jason Holder',
    'Alzarri Joseph'
]

role_check = (
    batter_eval[
        (batter_eval['striker_name'].isin(top_players)) &
        (batter_eval['pressure_level'] == 'High') &
        (batter_eval['runs_vs_expected'].notna())
    ]
    .groupby([
        'striker_name',
        'batting_order',
        'entry_phase'
    ])
    .agg(
        innings=('runs_vs_expected', 'count'),
        avg_runs=('runs', 'mean'),
        avg_above_expected=('runs_vs_expected', 'mean'),
        positive_pct=('runs_vs_expected',
                      lambda x: (x > 0).mean() * 100)
    )
    .reset_index()
    .sort_values(
        ['striker_name', 'innings'],
        ascending=[True, False]
    )
)

print(role_check.round(2).to_string(index=False))


        striker_name  batting_order entry_phase  innings  avg_runs  avg_above_expected  positive_pct
        Akeal Hosein             10       Death       12     10.25                6.68         66.67
        Akeal Hosein              8       Death        5      4.80               -2.19         20.00
        Akeal Hosein              8      Middle        5      7.20               -6.72         20.00
        Akeal Hosein              9       Death        2     17.00               12.00        100.00
        Akeal Hosein              9      Middle        2      8.50               -1.15         50.00
        Akeal Hosein             11       Death        1      1.00               -1.49          0.00
      Alzarri Joseph             10       Death       10      5.50                1.92         60.00
      Alzarri Joseph              9       Death        5      8.60                3.59         60.00
      Alzarri Joseph             10      Middle        2     11.50                4.53     

In [63]:

# 1. Pressure category-wise innings distribution
print("PRESSURE CATEGORY SUMMARY:")

print(
    batter_eval.groupby('pressure_level')
    .agg(
        innings=('runs', 'size'),
        eligible_innings=('runs_vs_expected', 'count'),
        avg_runs=('runs', 'mean'),
        avg_wickets_before=('wickets_before', 'mean')
    ).round(2)
)

# 2. Player-wise pressure category sample count
print("\nPLAYER PRESSURE SAMPLE DISTRIBUTION:")

print(
    pressure_analysis.pivot(
        index='striker_name',
        columns='pressure_level',
        values='innings'
    ).fillna(0).describe().round(2)
)

# 3. Top 15 players in medium + high pressure
high_pressure_players = (
    pressure_analysis[
        pressure_analysis['pressure_level'].isin(['Medium', 'High'])
    ]
    .groupby(['striker_id', 'striker_name'])
    .agg(
        pressure_innings=('innings', 'sum'),
        avg_above_expected=('avg_above_expected', 'mean'),
        positive_pct=('positive_pct', 'mean')
    )
    .reset_index()
)

high_pressure_players = high_pressure_players[
    high_pressure_players['pressure_innings'] >= 10
].sort_values('avg_above_expected', ascending=False)

print("\nTOP 15 MEDIUM + HIGH PRESSURE PLAYERS:")

print(
    high_pressure_players.head(15)
    .round(2).to_string(index=False)
)


PRESSURE CATEGORY SUMMARY:
                innings  eligible_innings  avg_runs  avg_wickets_before
pressure_level                                                         
High               2558              2553      6.53                7.09
Low                6483              6474     26.52                0.74
Medium             4081              4058     17.27                3.92

PLAYER PRESSURE SAMPLE DISTRIBUTION:


ValueError: Index contains duplicate entries, cannot reshape

In [64]:

player_pressure_distribution = (
    pressure_analysis
    .pivot_table(
        index='striker_name',
        columns='pressure_level',
        values='innings',
        aggfunc='sum',
        fill_value=0
    )
)

print("PLAYER PRESSURE SAMPLE DISTRIBUTION:")
print(player_pressure_distribution.to_string())


PLAYER PRESSURE SAMPLE DISTRIBUTION:
pressure_level           High  Low  Medium
striker_name                              
AHT Donald                  0    3       0
Aadil Ansari                0    0       2
Aamer Jamal                 2    0       1
Aamir Kaleem                0    3       0
Aarif Sheikh                0    0       1
Aaron Finch                 0   21       0
Aaron Hardie                1    4       6
Aaron Johnson               0    3       0
Aaron Jones                 0    3       0
Aasif Sheikh                0    2       0
Aayan Afzal Khan            1    0       2
Abbas Afridi               10    0       3
Abdul Basith                1    0       0
Abdul Samad                 2    3      41
Abdul Saqlain               0    0       2
Abdullah Ahmadzai           2    0       0
Abdullah Shafique           0    3       0
Abhijeet Tomar              0    1       0
Abhinav Manohar             0    4      16
Abhishek Sharma             0  124       2
Abishek Porel    

In [65]:

duplicate_check = (
    pressure_analysis
    .groupby(['striker_name', 'pressure_level'])
    .size()
    .reset_index(name='records')
)

print(
    duplicate_check[duplicate_check['records'] > 1]
    .sort_values('records', ascending=False)
    .head(20)
    .to_string(index=False)
)


striker_name pressure_level  records
 Abdul Samad         Medium        2


In [66]:

# Check duplicate player-name and player-ID combinations
name_id_check = (
    batter_eval[
        ['striker_id', 'striker_name']
    ]
    .drop_duplicates()
    .groupby('striker_name')
    .size()
    .reset_index(name='player_ids')
)

print("Names mapped to multiple player IDs:")
print(
    name_id_check[name_id_check['player_ids'] > 1]
    .sort_values('player_ids', ascending=False)
    .to_string(index=False)
)

print("\nTotal unique player IDs:",
      batter_eval['striker_id'].nunique())

print("Total unique player names:",
      batter_eval['striker_name'].nunique())


Names mapped to multiple player IDs:
striker_name  player_ids
 Abdul Samad           2
  Haider Ali           2

Total unique player IDs: 824
Total unique player names: 822


In [67]:

player_pressure_final = (
    batter_eval[
        batter_eval['runs_vs_expected'].notna()
    ]
    .groupby([
        'striker_id',
        'striker_name',
        'pressure_level'
    ])
    .agg(
        innings=('runs_vs_expected', 'count'),
        total_runs=('runs', 'sum'),
        avg_runs=('runs', 'mean'),
        avg_above_expected=('runs_vs_expected', 'mean'),
        median_above_expected=('runs_vs_expected', 'median'),
        positive_innings=('runs_vs_expected',
                          lambda x: (x > 0).sum())
    )
    .reset_index()
)

player_pressure_final['positive_pct'] = (
    player_pressure_final['positive_innings'] /
    player_pressure_final['innings'] * 100
).round(2)

print("FINAL PLAYER PRESSURE DATA:")
print("Rows:", len(player_pressure_final))
print("Unique player IDs:",
      player_pressure_final['striker_id'].nunique())

print("\nHIGH PRESSURE — MINIMUM 15 INNINGS")

high_final = (
    player_pressure_final[
        (player_pressure_final['pressure_level'] == 'High') &
        (player_pressure_final['innings'] >= 15)
    ]
    .sort_values('avg_above_expected', ascending=False)
)

print(
    high_final[
        ['striker_name', 'innings', 'total_runs',
         'avg_above_expected', 'median_above_expected',
         'positive_pct']
    ].head(15).round(2).to_string(index=False)
)


FINAL PLAYER PRESSURE DATA:
Rows: 1221
Unique player IDs: 824

HIGH PRESSURE — MINIMUM 15 INNINGS
        striker_name  innings  total_runs  avg_above_expected  median_above_expected  positive_pct
    Romario Shepherd       25         462                8.23                   8.12         68.00
Mahendra Singh Dhoni       17         267                7.59                   5.02         64.71
          Mark Adair       18         257                4.32                   1.52         55.56
       Faheem Ashraf       21         245                2.54                   0.10         57.14
         Rashid Khan       49         566                2.52                  -2.99         38.78
    Mitchell Santner       15         164                2.51                  -4.03         40.00
     Patrick Cummins       32         321                2.08                  -1.50         40.62
        Akeal Hosein       27         235                2.07                  -1.49         48.15
        Jas

In [68]:

print("DISMISSAL TYPES:")
print(
    complete_ball['dismissal_type']
    .value_counts(dropna=False)
    .to_string()
)

print("\nBOWLER DATA CHECK:")
print("Unique bowlers:", complete_ball['bowler_id'].nunique())
print("Total deliveries:", len(complete_ball))
print("Wickets recorded:", complete_ball['is_wicket'].sum())

print("\nBOWLER-CREDIT DISMISSALS:")
bowler_credit_types = [
    'Caught',
    'Bowled',
    'LBW',
    'Stump',
    'Hit Wicket'
]

print(
    complete_ball[
        complete_ball['dismissal_type'].isin(bowler_credit_types)
    ]['dismissal_type'].value_counts().to_string()
)


DISMISSAL TYPES:
dismissal_type
NaN                      182064
Caught                     7272
Bowled                     1676
LBW                         663
RunOut                      567
Stump                       198
Hit Wicket                   17
Retire Out                    6
Obstracting the field         1
Obstructing the field         1

BOWLER DATA CHECK:
Unique bowlers: 596
Total deliveries: 192465
Wickets recorded: 10401

BOWLER-CREDIT DISMISSALS:
dismissal_type
Caught        7272
Bowled        1676
LBW            663
Stump          198
Hit Wicket      17


In [69]:

bowler_data = complete_ball.copy()

bowler_credit_types = [
    'Caught',
    'Bowled',
    'LBW',
    'Stump',
    'Hit Wicket'
]

bowler_data['bowler_wicket'] = (
    bowler_data['dismissal_type']
    .isin(bowler_credit_types)
)

bowler_summary = (
    bowler_data
    .groupby(['bowler_id', 'bowler_name'])
    .agg(
        innings=('match_id', lambda x:
                 bowler_data.loc[x.index,
                 ['match_id', 'innings']].drop_duplicates().shape[0]),
        recorded_deliveries=('match_id', 'size'),
        batter_runs_conceded=('runs', 'sum'),
        wickets=('bowler_wicket', 'sum'),
        fours_conceded=('is_four', 'sum'),
        sixes_conceded=('is_six', 'sum'),
        dot_batter_runs=('runs', lambda x: (x == 0).sum())
    )
    .reset_index()
)

bowler_summary['boundary_conceded'] = (
    bowler_summary['fours_conceded'] +
    bowler_summary['sixes_conceded']
)

print("BOWLER SUMMARY:", bowler_summary.shape)

print(
    bowler_summary
    .sort_values('wickets', ascending=False)
    .head(20)
    .to_string(index=False)
)


BOWLER SUMMARY: (596, 10)
 bowler_id        bowler_name  innings  recorded_deliveries  batter_runs_conceded  wickets  fours_conceded  sixes_conceded  dot_batter_runs  boundary_conceded
      6464     Arshdeep Singh      146                 3414                  4621      192             531             173             1476                704
      4722        Rashid Khan      122                 2780                  3357      153             185             170             1018                355
      6256 Varun Chakravarthy       95                 2182                  2774      135             189             156              885                345
      4900  Wanindu Hasaranga       91                 2053                  2665      133             157             144              732                301
      1612   Yuzvendra Chahal       98                 2218                  2985      124             174             161              721                335
      7503       Rav

In [70]:

phase_data = complete_ball.copy()

phase_data['phase'] = np.select(
    [
        phase_data['overid'].between(1, 6),
        phase_data['overid'].between(7, 15),
        phase_data['overid'].between(16, 20)
    ],
    ['Powerplay', 'Middle', 'Death'],
    default='Unknown'
)

phase_data['bowler_wicket'] = (
    phase_data['dismissal_type'].isin([
        'Caught', 'Bowled', 'LBW',
        'Stump', 'Hit Wicket'
    ])
)

bowler_phase = (
    phase_data[phase_data['phase'] != 'Unknown']
    .groupby([
        'bowler_id',
        'bowler_name',
        'phase'
    ])
    .agg(
        innings=('match_id', lambda x:
            phase_data.loc[x.index,
            ['match_id', 'innings']].drop_duplicates().shape[0]),
        deliveries=('match_id', 'size'),
        batter_runs_conceded=('runs', 'sum'),
        wickets=('bowler_wicket', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum')
    )
    .reset_index()
)

bowler_phase['wickets_per_100_deliveries'] = (
    bowler_phase['wickets'] /
    bowler_phase['deliveries'] * 100
).round(2)

bowler_phase['boundaries'] = (
    bowler_phase['fours'] + bowler_phase['sixes']
)

print("PHASE-WISE BOWLING DATA:", bowler_phase.shape)

for phase in ['Powerplay', 'Middle', 'Death']:
    print(f"\n{'='*55}")
    print(f"{phase.upper()} — TOP 10 WICKET TAKERS")
    print('='*55)

    result = (
        bowler_phase[
            (bowler_phase['phase'] == phase) &
            (bowler_phase['innings'] >= 10)
        ]
        .sort_values('wickets', ascending=False)
        .head(10)
    )

    print(
        result[
            ['bowler_name', 'innings', 'deliveries',
             'wickets', 'wickets_per_100_deliveries',
             'boundaries']
        ].to_string(index=False)
    )


PHASE-WISE BOWLING DATA: (1493, 11)

POWERPLAY — TOP 10 WICKET TAKERS
       bowler_name  innings  deliveries  wickets  wickets_per_100_deliveries  boundaries
    Arshdeep Singh      145        1743       81                        4.65         392
 Bhuvneshwar Kumar      101        1373       63                        4.59         243
     Kagiso Rabada       68         903       47                        5.20         212
    Josh Hazlewood       64         790       46                        5.82         156
    Mohammed Siraj       83        1198       45                        3.76         250
    Mohammed Shami       57         883       43                        4.87         173
       Trent Boult       70         993       42                        4.23         198
      Jofra Archer       55         734       37                        5.04         173
Maheesh Theekshana       89         938       34                        3.62         169
       Jacob Duffy       39         509 

In [71]:

pressure_bowl = complete_ball.copy()

pressure_bowl = pressure_bowl.sort_values(
    ['match_id', 'innings', 'innings_ball_num']
)

pressure_bowl['wickets_before'] = (
    pressure_bowl.groupby(['match_id', 'innings'])['is_wicket']
    .cumsum()
    - pressure_bowl['is_wicket'].astype(int)
)

pressure_bowl['pressure_level'] = np.select(
    [
        pressure_bowl['wickets_before'].between(0, 2),
        pressure_bowl['wickets_before'].between(3, 5),
        pressure_bowl['wickets_before'].between(6, 9)
    ],
    ['Low', 'Medium', 'High'],
    default='Unknown'
)

pressure_bowl['bowler_wicket'] = (
    pressure_bowl['dismissal_type'].isin([
        'Caught', 'Bowled', 'LBW',
        'Stump', 'Hit Wicket'
    ])
)

pressure_bowl = pressure_bowl[
    pressure_bowl['pressure_level'] != 'Unknown'
]

group_cols = [
    'bowler_id', 'bowler_name', 'pressure_level'
]

bowler_pressure_summary = (
    pressure_bowl.groupby(group_cols)
    .agg(
        deliveries=('match_id', 'size'),
        batter_runs=('runs', 'sum'),
        wickets=('bowler_wicket', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum')
    )
    .reset_index()
)

pressure_innings = (
    pressure_bowl[
        group_cols + ['match_id', 'innings']
    ]
    .drop_duplicates()
    .groupby(group_cols)
    .size()
    .reset_index(name='innings')
)

bowler_pressure_summary = bowler_pressure_summary.merge(
    pressure_innings,
    on=group_cols,
    how='left'
)

bowler_pressure_summary['wickets_per_100'] = (
    bowler_pressure_summary['wickets'] /
    bowler_pressure_summary['deliveries'] * 100
).round(2)

bowler_pressure_summary['boundaries'] = (
    bowler_pressure_summary['fours'] +
    bowler_pressure_summary['sixes']
)

for level in ['High', 'Medium']:
    print(f"\n{'='*55}")
    print(f"{level.upper()} OPPOSITION WICKET SITUATIONS")
    print('='*55)

    result = (
        bowler_pressure_summary[
            (bowler_pressure_summary['pressure_level'] == level) &
            (bowler_pressure_summary['innings'] >= 10)
        ]
        .sort_values(
            ['wickets_per_100', 'wickets'],
            ascending=False
        )
        .head(15)
    )

    print(
        result[
            ['bowler_name', 'innings', 'deliveries',
             'wickets', 'wickets_per_100',
             'batter_runs', 'boundaries']
        ].to_string(index=False)
    )



HIGH OPPOSITION WICKET SITUATIONS
      bowler_name  innings  deliveries  wickets  wickets_per_100  batter_runs  boundaries
   Mitchell Starc       16         100       18            18.00          132          19
   Shardul Thakur       15         134       17            12.69          172          24
     Nitish Reddy       11          71        9            12.68           93          11
       Adam Milne       11          87       11            12.64          121          19
   Benjamin Sears       11          80       10            12.50           99          14
     Jason Holder       41         277       34            12.27          331          39
    Andre Russell       23         158       19            12.03          225          32
    Kuldeep Yadav       29         217       26            11.98          175          16
Jason Behrendorff       10          60        7            11.67           93          13
       Adam Zampa       19         131       15            11.45 

In [72]:

for level in ['High', 'Medium']:

    print(f"\n{'='*65}")
    print(f"{level.upper()} PRESSURE — SAMPLE SENSITIVITY")
    print('='*65)

    for min_innings, min_deliveries in [
        (10, 100),
        (15, 150),
        (20, 200)
    ]:

        result = bowler_pressure_summary[
            (bowler_pressure_summary['pressure_level'] == level) &
            (bowler_pressure_summary['innings'] >= min_innings) &
            (bowler_pressure_summary['deliveries'] >= min_deliveries)
        ].copy()

        result = result.sort_values(
            ['wickets_per_100', 'wickets'],
            ascending=False
        )

        print(
            f"\nMinimum innings: {min_innings} | "
            f"Minimum deliveries: {min_deliveries} | "
            f"Qualified bowlers: {len(result)}"
        )

        print(
            result[
                ['bowler_name', 'innings', 'deliveries',
                 'wickets', 'wickets_per_100']
            ].head(10).round(2).to_string(index=False)
        )



HIGH PRESSURE — SAMPLE SENSITIVITY

Minimum innings: 10 | Minimum deliveries: 100 | Qualified bowlers: 78
     bowler_name  innings  deliveries  wickets  wickets_per_100
  Mitchell Starc       16         100       18            18.00
  Shardul Thakur       15         134       17            12.69
    Jason Holder       41         277       34            12.27
   Andre Russell       23         158       19            12.03
   Kuldeep Yadav       29         217       26            11.98
      Adam Zampa       19         131       15            11.45
      Haris Rauf       24         184       21            11.41
Yuzvendra Chahal       26         174       19            10.92
      Obed McCoy       14         101       11            10.89
  Shaheen Afridi       26         204       22            10.78

Minimum innings: 15 | Minimum deliveries: 150 | Qualified bowlers: 42
      bowler_name  innings  deliveries  wickets  wickets_per_100
     Jason Holder       41         277       34      

In [73]:

MIN_INNINGS = 20
MIN_DELIVERIES = 200

final_bowling_rankings = {}

for level in ['High', 'Medium']:

    ranking = bowler_pressure_summary[
        (bowler_pressure_summary['pressure_level'] == level) &
        (bowler_pressure_summary['innings'] >= MIN_INNINGS) &
        (bowler_pressure_summary['deliveries'] >= MIN_DELIVERIES)
    ].copy()

    ranking = ranking.sort_values(
        ['wickets_per_100', 'wickets'],
        ascending=False
    ).reset_index(drop=True)

    ranking.insert(0, 'rank', range(1, len(ranking) + 1))

    final_bowling_rankings[level] = ranking

    print(f"\n{'='*60}")
    print(f"FINAL {level.upper()} PRESSURE BOWLING RANKING")
    print('='*60)

    print(
        ranking[
            ['rank', 'bowler_name', 'innings',
             'deliveries', 'wickets',
             'wickets_per_100', 'boundaries']
        ].head(15).round(2).to_string(index=False)
    )



FINAL HIGH PRESSURE BOWLING RANKING
 rank         bowler_name  innings  deliveries  wickets  wickets_per_100  boundaries
    1        Jason Holder       41         277       34            12.27          39
    2       Kuldeep Yadav       29         217       26            11.98          16
    3      Shaheen Afridi       26         204       22            10.78          23
    4   Mustafizur Rahman       34         314       31             9.87          38
    5         Rashid Khan       44         379       35             9.23          50
    6      Jasprit Bumrah       34         296       26             8.78          29
    7        Taskin Ahmed       23         200       17             8.50          31
    8       Harshal Patel       38         367       31             8.45          54
    9      Arshdeep Singh       70         534       44             8.24          89
   10  Varun Chakravarthy       37         281       23             8.19          31
   11 Matheesha Pathirana   

In [74]:

# Overall wicket-taking benchmark
overall_wickets = pressure_bowl['bowler_wicket'].sum()
overall_deliveries = len(pressure_bowl)

league_rate = overall_wickets / overall_deliveries

print("OVERALL PRESSURE BOWLING BENCHMARK")
print("Wickets:", overall_wickets)
print("Deliveries:", overall_deliveries)
print("Wickets per 100 deliveries:", round(league_rate * 100, 2))

# Shrinkage adjustment
# Prior strength = 100 recorded deliveries
PRIOR_DELIVERIES = 100

for level in ['High', 'Medium']:

    ranking = final_bowling_rankings[level].copy()

    ranking['adjusted_rate'] = (
        ranking['wickets'] +
        league_rate * PRIOR_DELIVERIES
    ) / (
        ranking['deliveries'] + PRIOR_DELIVERIES
    ) * 100

    ranking = ranking.sort_values(
        'adjusted_rate', ascending=False
    ).reset_index(drop=True)

    print(f"\n{'='*55}")
    print(f"{level.upper()} — SHRINKAGE-ADJUSTED RANKING")
    print('='*55)

    print(
        ranking[
            ['bowler_name', 'innings', 'deliveries',
             'wickets', 'wickets_per_100', 'adjusted_rate']
        ].head(15).round(2).to_string(index=False)
    )


OVERALL PRESSURE BOWLING BENCHMARK
Wickets: 9826
Deliveries: 192465
Wickets per 100 deliveries: 5.11

HIGH — SHRINKAGE-ADJUSTED RANKING
        bowler_name  innings  deliveries  wickets  wickets_per_100  adjusted_rate
       Jason Holder       41         277       34            12.27          10.37
      Kuldeep Yadav       29         217       26            11.98           9.81
     Shaheen Afridi       26         204       22            10.78           8.92
  Mustafizur Rahman       34         314       31             9.87           8.72
        Rashid Khan       44         379       35             9.23           8.37
     Jasprit Bumrah       34         296       26             8.78           7.85
     Arshdeep Singh       70         534       44             8.24           7.75
      Harshal Patel       38         367       31             8.45           7.73
 Varun Chakravarthy       37         281       23             8.19           7.38
       Taskin Ahmed       23         200    

In [75]:

PRIOR_DELIVERIES = 100

for level in ['High', 'Medium']:

    # Category-specific benchmark
    category_data = pressure_bowl[
        pressure_bowl['pressure_level'] == level
    ]

    category_wickets = category_data['bowler_wicket'].sum()
    category_deliveries = len(category_data)

    category_rate = category_wickets / category_deliveries

    ranking = bowler_pressure_summary[
        (bowler_pressure_summary['pressure_level'] == level) &
        (bowler_pressure_summary['innings'] >= 20) &
        (bowler_pressure_summary['deliveries'] >= 200)
    ].copy()

    # Category-specific shrinkage adjustment
    ranking['adjusted_rate'] = (
        ranking['wickets'] +
        category_rate * PRIOR_DELIVERIES
    ) / (
        ranking['deliveries'] + PRIOR_DELIVERIES
    ) * 100

    ranking = ranking.sort_values(
        'adjusted_rate', ascending=False
    ).reset_index(drop=True)

    ranking.insert(0, 'rank', range(1, len(ranking) + 1))

    print(f"\n{'='*60}")
    print(f"{level.upper()} PRESSURE — CATEGORY-SPECIFIC ADJUSTMENT")
    print('='*60)

    print("Category wickets:", category_wickets)
    print("Category deliveries:", category_deliveries)
    print("Category benchmark per 100:",
          round(category_rate * 100, 2))

    print(
        ranking[
            ['rank', 'bowler_name', 'innings',
             'deliveries', 'wickets',
             'wickets_per_100', 'adjusted_rate']
        ].head(15).round(2).to_string(index=False)
    )



HIGH PRESSURE — CATEGORY-SPECIFIC ADJUSTMENT
Category wickets: 1822
Category deliveries: 22985
Category benchmark per 100: 7.93
 rank         bowler_name  innings  deliveries  wickets  wickets_per_100  adjusted_rate
    1        Jason Holder       41         277       34            12.27          11.12
    2       Kuldeep Yadav       29         217       26            11.98          10.70
    3      Shaheen Afridi       26         204       22            10.78           9.84
    4   Mustafizur Rahman       34         314       31             9.87           9.40
    5         Rashid Khan       44         379       35             9.23           8.96
    6      Jasprit Bumrah       34         296       26             8.78           8.57
    7       Harshal Patel       38         367       31             8.45           8.34
    8        Taskin Ahmed       23         200       17             8.50           8.31
    9      Arshdeep Singh       70         534       44             8.24       

In [76]:

for level in ['High', 'Medium']:

    category_data = pressure_bowl[
        pressure_bowl['pressure_level'] == level
    ]

    benchmark = (
        category_data['bowler_wicket'].sum() /
        len(category_data)
    )

    print(f"\n{'='*65}")
    print(f"{level.upper()} — PRIOR STRENGTH SENSITIVITY")
    print('='*65)

    eligible = bowler_pressure_summary[
        (bowler_pressure_summary['pressure_level'] == level) &
        (bowler_pressure_summary['innings'] >= 20) &
        (bowler_pressure_summary['deliveries'] >= 200)
    ].copy()

    for prior in [50, 100, 200]:

        eligible['adjusted_rate'] = (
            eligible['wickets'] + benchmark * prior
        ) / (
            eligible['deliveries'] + prior
        ) * 100

        result = eligible.sort_values(
            'adjusted_rate', ascending=False
        ).head(10)

        print(f"\nPrior deliveries: {prior}")

        print(
            result[
                ['bowler_name', 'innings',
                 'wickets', 'adjusted_rate']
            ].round(2).to_string(index=False)
        )



HIGH — PRIOR STRENGTH SENSITIVITY

Prior deliveries: 50
       bowler_name  innings  wickets  adjusted_rate
      Jason Holder       41       34          11.61
     Kuldeep Yadav       29       26          11.22
    Shaheen Afridi       26       22          10.22
 Mustafizur Rahman       34       31           9.61
       Rashid Khan       44       35           9.08
    Jasprit Bumrah       34       26           8.66
      Taskin Ahmed       23       17           8.39
     Harshal Patel       38       31           8.38
    Arshdeep Singh       70       44           8.21
Varun Chakravarthy       37       23           8.15

Prior deliveries: 100
       bowler_name  innings  wickets  adjusted_rate
      Jason Holder       41       34          11.12
     Kuldeep Yadav       29       26          10.70
    Shaheen Afridi       26       22           9.84
 Mustafizur Rahman       34       31           9.40
       Rashid Khan       44       35           8.96
    Jasprit Bumrah       34       26

In [77]:

import os
os.makedirs("pressure_player_results", exist_ok=True)

# BATTER RANKINGS
for level in ['High', 'Medium']:

    batter_result = (
        batter_eval[
            (batter_eval['pressure_level'] == level) &
            (batter_eval['runs_vs_expected'].notna())
        ]
        .groupby(['striker_id', 'striker_name'])
        .agg(
            innings=('runs_vs_expected', 'count'),
            total_runs=('runs', 'sum'),
            avg_above_expected=('runs_vs_expected', 'mean'),
            median_above_expected=('runs_vs_expected', 'median'),
            positive_innings=('runs_vs_expected',
                              lambda x: (x > 0).sum())
        )
        .reset_index()
    )

    batter_result['positive_pct'] = (
        batter_result['positive_innings'] /
        batter_result['innings'] * 100
    )

    batter_result = (
        batter_result[batter_result['innings'] >= 10]
        .sort_values('avg_above_expected', ascending=False)
    )

    batter_result.to_csv(
        f"pressure_player_results/batting_{level.lower()}.csv",
        index=False
    )

# BOWLER RANKINGS
for level in ['High', 'Medium']:

    category_data = pressure_bowl[
        pressure_bowl['pressure_level'] == level
    ]

    benchmark = (
        category_data['bowler_wicket'].sum() /
        len(category_data)
    )

    bowling_result = bowler_pressure_summary[
        (bowler_pressure_summary['pressure_level'] == level) &
        (bowler_pressure_summary['innings'] >= 20) &
        (bowler_pressure_summary['deliveries'] >= 200)
    ].copy()

    bowling_result['adjusted_rate'] = (
        (bowling_result['wickets'] + benchmark * 100) /
        (bowling_result['deliveries'] + 100) * 100
    )

    bowling_result = bowling_result.sort_values(
        'adjusted_rate', ascending=False
    )

    bowling_result.to_csv(
        f"pressure_player_results/bowling_{level.lower()}.csv",
        index=False
    )

print("FINAL FILES SAVED:")
print(os.listdir("pressure_player_results"))


FINAL FILES SAVED:
['bowling_high.csv', 'batting_medium.csv', 'batting_high.csv', 'bowling_medium.csv']


In [78]:
import pandas as pd
import os

folder = "pressure_player_results"

for file in sorted(os.listdir(folder)):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        print("\n" + "="*65)
        print(file.upper())
        print("="*65)
        print(df.head(10).to_string(index=False))


BATTING_HIGH.CSV
 striker_id         striker_name  innings  total_runs  avg_above_expected  median_above_expected  positive_innings  positive_pct
       6726     Romario Shepherd       25         462            8.227205               8.123894                17     68.000000
          8 Mahendra Singh Dhoni       17         267            7.586889               5.024233                11     64.705882
       2382        Jamie Overton       10         155            5.961689               4.568831                 6     60.000000
       3629         Corbin Bosch       11         164            5.599967               4.022617                 7     63.636364
       5379           Mark Adair       18         257            4.321438               1.518578                10     55.555556
       1813        Deepak Chahar       11          94            2.649270              -3.659459                 5     45.454545
       5189        Faheem Ashraf       21         245            2.539804      